P(True)

In [ ]:
!pip install -U transformers trl peft bitsandbytes datasets accelerate rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, json, numpy as np, torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_ptrue"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_JSON  = f"{DATA_DIR}/dpo_test_questions.json"
TRAIN_JSON = f"{DATA_DIR}/dpo_train_questions.json"
FULL_JSON  = f"{DATA_DIR}/hedge_pre_rl_1000_full.json"
JUDGE_JSON = f"{DATA_DIR}/rejudged_1000.json"

RETRIEVE_K = 10
MAX_TOK    = 220
BASE_ACC   = 0.515
N_VAL      = 150

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

INSTRUCTION = ("Answer the question using the evidence, reasoning step by step. "
               "Finish with 'Therefore, the answer is X.'")


In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                          device_map="auto").eval()

full   = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids  = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
  
    train_ids = [q for q in full if q not in set(test_ids)]
val_ids = [q for q in train_ids if q in full][:N_VAL]
print(f"test={len(test_ids)}  val={len(val_ids)}")

base_judge = {int(x["question_index"]): bool(x["judge_correct"])
              for x in json.load(open(JUDGE_JSON))}

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
q2ex = {e["question"]: e for e in ds}

def tk(s): return re.findall(r"\w+", s.lower())
def all_sents(ex): return [s.strip() for p in ex["context"]["sentences"] for s in p if s.strip()]
def retrieve(q, ex):
    sents = all_sents(ex)
    if not sents: return []
    bm = BM25Okapi([tk(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(tk(q)))[::-1][:RETRIEVE_K]]


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [ ]:
def answer_prompt(question, evidence):
    ev = "\n".join(f"- {e}" for e in evidence)
    return f"{INSTRUCTION}\n\nEvidence:\n{ev}\n\nQuestion: {question}"

def generate_answer(question, evidence) -> str:
    msg = answer_prompt(question, evidence)
    inp = tok.apply_chat_template([{"role": "user", "content": msg}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=MAX_TOK, do_sample=False,
                          pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]

_TRUE_IDS = None
_FALSE_IDS = None
def _tf_token_ids():
    global _TRUE_IDS, _FALSE_IDS
    if _TRUE_IDS is None:
        
        cand_true  = ["True", " True", "true", " true", "T", " T"]
        cand_false = ["False", " False", "false", " false", "F", " F"]
        _TRUE_IDS  = list({tok(t, add_special_tokens=False)["input_ids"][0] for t in cand_true
                           if tok(t, add_special_tokens=False)["input_ids"]})
        _FALSE_IDS = list({tok(f, add_special_tokens=False)["input_ids"][0] for f in cand_false
                           if tok(f, add_special_tokens=False)["input_ids"]})
    return _TRUE_IDS, _FALSE_IDS

def p_true(question, evidence, answer) -> float:
    """Kadavath-style P(True): probability the model assigns to 'True' for its own answer."""
    ev = "\n".join(f"- {e}" for e in evidence)
    prompt = (f"Evidence:\n{ev}\n\nQuestion: {question}\n"
              f"Proposed answer: {answer}\n\n"
              "Is the proposed answer correct given the evidence? "
              "Respond with a single word: True or False.\nAnswer:")
    inp = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=1, do_sample=False,
                          output_scores=True, return_dict_in_generate=True,
                          pad_token_id=tok.pad_token_id)
    logits = out.scores[0][0]                       # first generated token logits
    probs = torch.softmax(logits.float(), dim=-1)
    tids, fids = _tf_token_ids()
    p_t = float(sum(probs[i] for i in tids))
    p_f = float(sum(probs[i] for i in fids))
    denom = p_t + p_f
    return p_t / denom if denom > 1e-9 else 0.5     # normalised P(True)


In [ ]:
def run_split(ids, tag):
    recs = []
    for j, qi in enumerate(ids):
        r = full.get(qi); ex = q2ex.get(r["question"]) if r else None
        if not r or not ex:
            continue
        ev = retrieve(r["question"], ex)
        text = generate_answer(r["question"], ev)
        ans = extract_answer(text)
        conf = p_true(r["question"], ev, ans)
        recs.append({"qi": qi, "question": r["question"], "gold": r["gold_answer"],
                     "answer": ans, "p_true": conf})
        if (j + 1) % 25 == 0:
            print(f"  {tag}: {j+1}/{len(ids)}")
            json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json", "w"))
    json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json", "w"))
    return recs

print("=== generating VALIDATION (for threshold) ===")
val_recs = run_split(val_ids, "val")
print("=== generating TEST ===")
test_recs = run_split(test_ids, "test")

=== generating VALIDATION (for threshold) ===
  val: 25/150
  val: 50/150
  val: 75/150
  val: 100/150
  val: 125/150
  val: 150/150
=== generating TEST ===
  test: 25/200
  test: 50/200
  test: 75/200
  test: 100/200
  test: 125/200
  test: 150/200
  test: 175/200
  test: 200/200


In [ ]:
def judge(question, gold, ans):
    if not ans: return False
    p = (f"Question: {question}\nGold answer: {gold}\nPredicted: {ans}\n"
         "Is the predicted answer correct? Accept paraphrases/equivalent. "
         "Reply exactly yes or no.")
    r = oai.chat.completions.create(model="gpt-4o-mini", temperature=0, max_tokens=4,
                                    messages=[{"role": "user", "content": p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

for recs in (val_recs, test_recs):
    for x in recs:
        if "correct" not in x:
            x["correct"] = judge(x["question"], x["gold"], x["answer"])
json.dump(val_recs,  open(f"{OUT_DIR}/val_records.json", "w"))
json.dump(test_recs, open(f"{OUT_DIR}/test_records.json", "w"))

def ths_from_counts(cc, cw, over, N, base=BASE_ACC):
    THS = ((cc/N)*(1-base) - (cw/N)*base)/(1-base)*100
    cov = (cc + cw) / N
    sel = cc / (cc + cw) if (cc + cw) else 0.0
    cerr = cw / N
    oab = over / N   
    return THS, cov, sel, cerr, oab

def evaluate(recs, thr):
    cc = cw = ab = over = 0
    N = len(recs)
    for x in recs:
        if x["p_true"] < thr:             
            ab += 1
            if base_judge.get(x["qi"], False): 
                over += 1
        else:                              
            if x["correct"]: cc += 1
            else: cw += 1
    ths, cov, sel, cerr, oab = ths_from_counts(cc, cw, over, N)
    return {"thr": thr, "cc": cc, "cw": cw, "ab": ab, "over": over,
            "cov": cov, "sel_acc": sel, "conf_err": cerr, "over_abst": oab, "THS": ths}

# tune threshold on VALIDATION: sweep, pick max THS
grid = [round(t, 2) for t in np.arange(0.30, 0.96, 0.02)]
val_scores = [evaluate(val_recs, t) for t in grid]
best = max(val_scores, key=lambda d: d["THS"])
best_thr = best["thr"]
print(f"\nbest validation threshold: P(True) >= {best_thr}  (val THS {best['THS']:.2f})")

test_result = evaluate(test_recs, best_thr)
print("\n=== Calibrated P(True) on HotpotQA TEST (base 0.515) ===")
for k, v in test_result.items():
    print(f"  {k}: {v}")

json.dump({"best_thr": best_thr, "val_curve": val_scores, "test": test_result},
          open(f"{OUT_DIR}/ptrue_result.json", "w"))
print("\nsaved -> ptrue_result.json")




best validation threshold: P(True) >= 0.94  (val THS 25.55)

=== Calibrated P(True) on HotpotQA TEST (base 0.515) ===
  thr: 0.94
  cc: 70
  cw: 11
  ab: 119
  over: 41
  cov: 0.405
  sel_acc: 0.8641975308641975
  conf_err: 0.055
  over_abst: 0.205
  THS: 29.159793814432987

saved -> ptrue_result.json
NOTE: replace ths_from_counts with your exact run_ths formula for the final table.


log probe and verbalized


In [ ]:
import os, re, json, numpy as np, torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_logprob_verbalized"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_JSON  = f"{DATA_DIR}/dpo_test_questions.json"
TRAIN_JSON = f"{DATA_DIR}/dpo_train_questions.json"
FULL_JSON  = f"{DATA_DIR}/hedge_pre_rl_1000_full.json"
JUDGE_JSON = f"{DATA_DIR}/rejudged_1000.json"

RETRIEVE_K = 10
MAX_TOK    = 220
BASE_ACC   = 0.515
N_VAL      = 150

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

INSTRUCTION = ("Answer the question using the evidence, reasoning step by step. "
               "Finish with 'Therefore, the answer is X.'")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                          device_map="auto").eval()

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in set(test_ids)]
val_ids = [q for q in train_ids if q in full][:N_VAL]
base_judge = {int(x["question_index"]): bool(x["judge_correct"]) for x in json.load(open(JUDGE_JSON))}
print(f"test={len(test_ids)} val={len(val_ids)}")

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
q2ex = {e["question"]: e for e in ds}
def tk(s): return re.findall(r"\w+", s.lower())
def all_sents(ex): return [s.strip() for p in ex["context"]["sentences"] for s in p if s.strip()]
def retrieve(q, ex):
    sents = all_sents(ex)
    if not sents: return []
    bm = BM25Okapi([tk(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(tk(q)))[::-1][:RETRIEVE_K]]
def ev_block(ex, q): return "\n".join(f"- {e}" for e in retrieve(q, ex))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [ ]:
def gen_with_logprobs(q, evb):
    """Greedy generation; returns text, per-token logprobs, token ids (for span scoring)."""
    prompt = f"{INSTRUCTION}\n\nEvidence:\n{evb}\n\nQuestion: {q}"
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    out = lm.generate(**inp, max_new_tokens=MAX_TOK, do_sample=False,
                      pad_token_id=tok.pad_token_id, output_scores=True,
                      return_dict_in_generate=True)
    seq = out.sequences[0][inp["input_ids"].shape[1]:]
    lps = [float(torch.log_softmax(s[0].float(), -1)[t]) for s, t in zip(out.scores, seq)]
    ids = seq.tolist()
    text = tok.decode(seq, skip_special_tokens=True).strip()
    return text, lps, ids

def answer_span_logprobs(text, lps, ids):
    """THE LENGTH-CONFOUND FIX: score only the answer span (after 'the answer is'),
    not the whole chain (which is dominated by length)."""
    low = text.lower()
    marker = low.rfind("the answer is")
    if marker == -1:
        return lps[-20:] if len(lps) > 20 else lps
    target_char = marker + len("the answer is")
    start_tok = None
    for i in range(len(ids)):
        if len(tok.decode(ids[:i+1], skip_special_tokens=True)) >= target_char:
            start_tok = i; break
    if start_tok is None:
        return lps[-20:] if len(lps) > 20 else lps
    return lps[start_tok:] if lps[start_tok:] else lps[-5:]

def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]

In [ ]:
def verbalized_confidence(q, ans, evb):
    """Tian et al. 2023: ask the model to state a 0-100 confidence in its answer."""
    prompt = (f"Evidence:\n{evb}\n\nQuestion: {q}\nProposed answer: {ans}\n\n"
              "How confident are you that this answer is correct, given the evidence? "
              "Respond with ONLY a number from 0 to 100 (0 = certainly wrong, 100 = certainly correct).")
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=6, do_sample=False, pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\d{1,3}", txt)
    if not m: return 0.5
    v = min(100, max(0, int(m.group(0))))
    return v / 100.0

In [ ]:
def run_split(ids, tag):
    recs = []
    for j, qi in enumerate(ids):
        r = full.get(qi); ex = q2ex.get(r["question"]) if r else None
        if not r or not ex: continue
        evb = ev_block(ex, r["question"])
        text, lps, tids = gen_with_logprobs(r["question"], evb)
        ans = extract_answer(text)
        span = answer_span_logprobs(text, lps, tids)
        logprob_score = float(np.mean(span)) if span else -1e9   # mean answer-span logprob (length-normalized)
        vconf = verbalized_confidence(r["question"], ans, evb)
        recs.append({"qi": qi, "question": r["question"], "gold": r["gold_answer"],
                     "answer": ans, "logprob": logprob_score, "verbalized": vconf})
        if (j+1) % 25 == 0:
            print(f"  {tag}: {j+1}/{len(ids)}")
            json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json", "w"))
    json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json", "w"))
    return recs

print("=== VALIDATION ===")
val_recs = run_split(val_ids, "val")
print("=== TEST ===")
test_recs = run_split(test_ids, "test")

=== VALIDATION ===
  val: 25/150
  val: 50/150
  val: 75/150
  val: 100/150
  val: 125/150
  val: 150/150
=== TEST ===
  test: 25/200
  test: 50/200
  test: 75/200
  test: 100/200
  test: 125/200
  test: 150/200
  test: 175/200
  test: 200/200


In [ ]:
def judge(q, gold, ans):
    if not ans: return False
    p = (f"Question: {q}\nGold answer: {gold}\nPredicted: {ans}\n"
         "Is the predicted answer correct? Accept paraphrases/equivalent. Reply yes or no.")
    r = oai.chat.completions.create(model="gpt-4o-mini", temperature=0, max_tokens=4,
                                    messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

for recs in (val_recs, test_recs):
    for x in recs:
        if "correct" not in x:
            x["correct"] = judge(x["question"], x["gold"], x["answer"])
json.dump(val_recs, open(f"{OUT_DIR}/val_records.json","w"))
json.dump(test_recs, open(f"{OUT_DIR}/test_records.json","w"))

def ths(cc, cw, N, base=BASE_ACC):
    return ((cc/N)*(1-base) - (cw/N)*base)/(1-base)*100

def evaluate(recs, signal, thr):
    """abstain if confidence < thr (higher signal = more confident for both)."""
    cc=cw=ab=over=0; N=len(recs)
    for x in recs:
        conf = x[signal]
        if conf < thr:
            ab += 1
            if base_judge.get(x["qi"], False): over += 1
        else:
            if x["correct"]: cc += 1
            else: cw += 1
    return {"thr":thr,"cc":cc,"cw":cw,"ab":ab,"over":over,"cov":(cc+cw)/N,
            "sel_acc":cc/(cc+cw) if (cc+cw) else 0,"conf_err":cw/N,
            "over_abst":over/N,"THS":ths(cc,cw,N)}

def tune_and_eval(signal, name):
    vals = sorted(x[signal] for x in val_recs)
    grid = [vals[int(f*len(vals))] for f in np.arange(0.05, 0.96, 0.05)]
    val_scores = [evaluate(val_recs, signal, t) for t in grid]
    best = max(val_scores, key=lambda d: d["THS"])
    test_result = evaluate(test_recs, signal, best["thr"])
    print(f"\n=== {name} (base 0.515) ===")
    print(f"  best val threshold: {signal} >= {best['thr']:.4f}  (val THS {best['THS']:.2f})")
    for k,v in test_result.items(): print(f"  {k}: {v}")
    return {"best_thr":best["thr"], "val_curve":val_scores, "test":test_result}

res_logprob = tune_and_eval("logprob", "Logprob threshold (Jurayj/Ren)")
res_verbal  = tune_and_eval("verbalized", "Verbalized confidence (Tian et al.)")

json.dump({"logprob":res_logprob, "verbalized":res_verbal},
          open(f"{OUT_DIR}/logprob_verbalized_results.json","w"))
print("\n" + "="*54)
print(f"  Logprob threshold:      THS {res_logprob['test']['THS']:.2f}")
print(f"  Verbalized confidence:  THS {res_verbal['test']['THS']:.2f}")
print("  (compare: P(True) 29.16 | R-Tuning 15.31 | rule 44.63)")
print("="*54)



=== Logprob threshold (Jurayj/Ren) (base 0.515) ===
  best val threshold: logprob >= -0.4230  (val THS 33.48)
  thr: -0.4230025142392833
  cc: 127
  cw: 59
  ab: 14
  over: 5
  cov: 0.93
  sel_acc: 0.6827956989247311
  conf_err: 0.295
  over_abst: 0.025
  THS: 32.17525773195876

=== Verbalized confidence (Tian et al.) (base 0.515) ===
  best val threshold: verbalized >= 0.0000  (val THS 36.77)
  thr: 0.0
  cc: 133
  cw: 67
  ab: 0
  over: 0
  cov: 1.0
  sel_acc: 0.665
  conf_err: 0.335
  over_abst: 0.0
  THS: 30.927835051546392

  Logprob threshold:      THS 32.18
  Verbalized confidence:  THS 30.93
  (compare: P(True) 29.16 | R-Tuning 15.31 | rule 44.63)


In [ ]:
import json, numpy as np
D = "/content/drive/MyDrive/hedge_run/baseline_logprob_verbalized"
test = json.load(open(f"{D}/test_records.json"))
v = [x["verbalized"] for x in test]
print(f"verbalized confidence distribution:")
print(f"  min={min(v):.2f} max={max(v):.2f} mean={np.mean(v):.2f} median={np.median(v):.2f}")
print(f"  fraction >= 0.9: {sum(1 for x in v if x>=0.9)/len(v):.2f}")
print(f"  fraction >= 0.8: {sum(1 for x in v if x>=0.8)/len(v):.2f}")
print(f"  unique values: {sorted(set(v))[:15]}")
# does it separate correct from wrong at all?
cor = [x["verbalized"] for x in test if x["correct"]]
wr  = [x["verbalized"] for x in test if not x["correct"]]
print(f"  mean verbalized | correct: {np.mean(cor):.3f}")
print(f"  mean verbalized | wrong:   {np.mean(wr):.3f}")

verbalized confidence distribution:
  min=0.00 max=1.00 mean=0.69 median=0.85
  fraction >= 0.9: 0.16
  fraction >= 0.8: 0.79
  unique values: [0.0, 0.2, 0.5, 0.6, 0.8, 0.85, 0.95, 1.0]
  mean verbalized | correct: 0.790
  mean verbalized | wrong:   0.501


In [ ]:
import json, numpy as np
from sklearn.metrics import roc_auc_score
D = "/content/drive/MyDrive/hedge_run/baseline_logprob_verbalized"
test = json.load(open(f"{D}/test_records.json"))
y_wrong = [0 if x["correct"] else 1 for x in test]

auc_verbal = roc_auc_score(y_wrong, [1 - x["verbalized"] for x in test])
auc_logprob = roc_auc_score(y_wrong, [-x["logprob"] for x in test])  
print(f"Verbalized confidence AUROC (predicting wrong): {auc_verbal:.3f}")
print(f"Logprob AUROC (predicting wrong): {auc_logprob:.3f}")


Verbalized confidence AUROC (predicting wrong): 0.714
Logprob AUROC (predicting wrong): 0.666
  (compare: P(True) ~?, semantic entropy 0.60, your self_consistency 0.596)


**CRaFT** 2wiki

In [ ]:
import os, re, json, numpy as np, torch
from rank_bm25 import BM25Okapi
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)
from sentence_transformers import SentenceTransformer
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_craft_2wiki_fixed"
ADAPTER    = f"{OUT_DIR}/craft_adapter"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON  = f"{DATA_DIR}/2wiki_full.json"
TEST_JSON  = f"{DATA_DIR}/2wiki_test_questions.json"
TRAIN_JSON = f"{DATA_DIR}/2wiki_train_questions.json"
FAIR_JSON  = f"{DATA_DIR}/2wiki_base_point_fair.json"
SAMPLES_CACHE = f"{OUT_DIR}/train_samples.json"
STATE_CACHE   = f"{OUT_DIR}/knowledge_state.json"

RETRIEVE_K = 10
BASE_ACC   = 0.440
TAU_MU     = 0.5
RATIO_IDK  = 4
K_SAMPLES  = 8          # MATCHED to HotpotQA
SAMPLE_TEMP = 0.7       # MATCHED to HotpotQA
EMBED_ID   = "sentence-transformers/all-MiniLM-L6-v2"
N_TRAIN_STATE = 800
GEN_MAX    = 64         # direct answers are short

INSTRUCTION = ("Answer the question based on the evidence. Give a brief, direct answer. "
               "If the evidence is insufficient to answer, respond exactly with \"I don't know.\"")
IDK_TEXT = "I don't know."

NON_ANSWER = ["cannot determine","cannot be determined","could not determine","does not allow",
    "we cannot","do not have","information provided does not","there is no movie",
    "no movie in the given","cannot definitively","x, where x","x (where","does not provide",
    "not allow us to determine","cannot be determined from","does not contain","cannot be precisely",
    "not confident","cannot verify","unable to determine","insufficient","does not specify","cannot answer"]
def is_nonanswer(a): a=str(a).lower(); return any(m in a for m in NON_ANSWER)

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = set(int(x["question_index"]) for x in json.load(open(TEST_JSON)))
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in test_ids]
train_ids = [q for q in train_ids if q in full and q not in test_ids][:N_TRAIN_STATE]

fair = json.load(open(FAIR_JSON))
base_correct_q = {int(k): bool(v) for k, v in fair["per_q"].items()}
n_base_correct = sum(base_correct_q.values())
print(f"train: {len(train_ids)}  test: {len(test_ids)}  base_correct={n_base_correct} (expect 88)")

def tk(s): return re.findall(r"\w+", s.lower())
def all_sents(rec): return [s.strip() for p in rec["context"]["sentences"] for s in p if s and s.strip()]
def retrieve(q, rec):
    sents=all_sents(rec)
    if not sents: return []
    bm=BM25Okapi([tk(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(tk(q)))[::-1][:RETRIEVE_K]]
def build_prompt(q, rec):
    ev="\n".join(f"- {e}" for e in retrieve(q, rec))
    return f"{INSTRUCTION}\n\nEvidence:\n{ev}\n\nQuestion: {q}"


train: 800  test: 200  base_correct=88 (expect 88)


In [ ]:
tok=AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token=tok.eos_token
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16)
lm=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb,device_map="auto").eval()

def sample_k(question, rec, k=K_SAMPLES):
    prompt=build_prompt(question, rec)
    inp=tok.apply_chat_template([{"role":"user","content":prompt}],add_generation_prompt=True,
                                return_tensors="pt",return_dict=True).to(lm.device)
    with torch.no_grad():
        out=lm.generate(**inp,max_new_tokens=GEN_MAX,do_sample=True,temperature=SAMPLE_TEMP,
                        top_p=0.9,num_return_sequences=k,pad_token_id=tok.pad_token_id)
    plen=inp["input_ids"].shape[1]
    return [tok.decode(s[plen:],skip_special_tokens=True).strip().rstrip(".") for s in out]

samples_store=json.load(open(SAMPLES_CACHE)) if os.path.exists(SAMPLES_CACHE) else {}
for i,qi in enumerate(train_ids):
    if str(qi) in samples_store: continue
    r=full.get(qi)
    if not r: continue
    samples_store[str(qi)]=sample_k(r["question"],r)
    if (i+1)%20==0: print(f"  samples {i+1}/{len(train_ids)}"); json.dump(samples_store,open(SAMPLES_CACHE,"w"))
json.dump(samples_store,open(SAMPLES_CACHE,"w"))
print(f"generated samples for {len(samples_store)} train questions")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generated samples for 800 train questions


In [ ]:
embedder=SentenceTransformer(EMBED_ID)
def judge_one(q,gold,ans):
    if not ans or not ans.strip() or is_nonanswer(ans): return False
    p=(f"Question: {q}\nGold: {gold}\nPrediction: {ans}\nCorrect? Accept paraphrases. yes/no.")
    r=oai.chat.completions.create(model="gpt-4o-mini",temperature=0,max_tokens=4,
                                  messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")
def certainty(samples):
    s=[x for x in samples if x and x.strip()]
    if len(s)<2: return 1.0
    E=embedder.encode(s,normalize_embeddings=True); S=E@E.T; n=len(s)
    return float((S.sum()-np.trace(S))/(n*(n-1)))
def majority(samples):
    from collections import Counter
    s=[x for x in samples if x and x.strip()]
    return Counter(s).most_common(1)[0][0] if s else ""

if os.path.exists(STATE_CACHE):
    state={int(k):v for k,v in json.load(open(STATE_CACHE)).items()}
    print(f"loaded knowledge state: {len(state)}")
else:
    state={}
    for i,qi in enumerate(train_ids):
        r=full.get(qi)
        if not r or str(qi) not in samples_store: continue
        gold=r.get("gold_answer",""); samples=samples_store[str(qi)]
        mu=float(np.mean([judge_one(r["question"],gold,a) for a in samples])) if samples else 0.0
        sigma=certainty(samples)
        state[qi]={"mu":mu,"sigma":sigma,"gold":gold,"question":r["question"],
                   "majority_answer":majority(samples)}
        if (i+1)%50==0: print(f"  state {i+1}"); json.dump({str(k):v for k,v in state.items()},open(STATE_CACHE,"w"))
    json.dump({str(k):v for k,v in state.items()},open(STATE_CACHE,"w"))
    print(f"knowledge state: {len(state)}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

loaded knowledge state: 800


In [ ]:
van_cand=[(qi,s) for qi,s in state.items() if s["mu"]>=TAU_MU]
idk_cand=[(qi,s) for qi,s in state.items() if s["mu"]< TAU_MU]
van_cand.sort(key=lambda kv:kv[1]["sigma"],reverse=True)
idk_cand.sort(key=lambda kv:kv[1]["sigma"])
N_van=max(1,min(len(van_cand),len(idk_cand)//RATIO_IDK)) if idk_cand else len(van_cand)
N_idk=min(len(idk_cand),N_van*RATIO_IDK)
van_sel,idk_sel=van_cand[:N_van],idk_cand[:N_idk]
print(f"CorCer-RAIT: {len(van_sel)} vanilla + {len(idk_sel)} IdK (1:{RATIO_IDK})")

train_rows=[]
for qi,s in van_sel:
    r=full.get(qi)
    if not r: continue
    train_rows.append({"prompt":build_prompt(s["question"],r),"completion":f" {s['majority_answer']}"})
for qi,s in idk_sel:
    r=full.get(qi)
    if not r: continue
    train_rows.append({"prompt":build_prompt(s["question"],r),"completion":f" {IDK_TEXT}"})
np.random.shuffle(train_rows)
json.dump(train_rows,open(f"{OUT_DIR}/craft_train_data.json","w"))
print(f"total SFT examples: {len(train_rows)}")
print(f"  sample vanilla: {repr([r for r in train_rows if IDK_TEXT not in r['completion']][0]['completion'])}")
print(f"  sample IdK:     {repr([r for r in train_rows if IDK_TEXT in r['completion']][0]['completion'])}")


CorCer-RAIT: 50 vanilla + 200 IdK (1:4)
total SFT examples: 250
  sample vanilla: " I don't know"
  sample IdK:     " I don't know."


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb,device_map="auto")
peft_cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",task_type="CAUSAL_LM",
                    target_modules=["q_proj","k_proj","v_proj","o_proj"])
def fmt(ex):
    text=tok.apply_chat_template([{"role":"user","content":ex["prompt"]}],tokenize=False,add_generation_prompt=True)
    return {"text":text+ex["completion"]+tok.eos_token}
sft_ds=Dataset.from_list(train_rows).map(fmt)
cfg=SFTConfig(output_dir=OUT_DIR,per_device_train_batch_size=2,gradient_accumulation_steps=8,
              num_train_epochs=3,learning_rate=1e-5,lr_scheduler_type="cosine",bf16=True,
              logging_steps=10,save_strategy="no",report_to="none")
trainer=SFTTrainer(model=model,args=cfg,train_dataset=sft_ds,peft_config=peft_cfg)
trainer.train(); trainer.save_model(ADAPTER)
print("CRaFT SFT complete.")



Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.388195
20,2.837792
30,2.789702
40,2.403934


CRaFT SFT complete.


In [ ]:
from peft import PeftModel
m=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb,device_map="auto")
m=PeftModel.from_pretrained(m,ADAPTER).eval()

def is_refusal(t):
    tl=t.lower().strip()
    return ("i don't know" in tl) or ("i do not know" in tl) or is_nonanswer(tl)
def generate(prompt):
    inp=tok.apply_chat_template([{"role":"user","content":prompt}],add_generation_prompt=True,
                                return_tensors="pt",return_dict=True).to(m.device)
    with torch.no_grad():
        out=m.generate(**inp,max_new_tokens=GEN_MAX,do_sample=False,pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:],skip_special_tokens=True).strip()
def judge(q,gold,ans):
    if not ans or not ans.strip(): return False
    p=(f"Question: {q}\nGold: {gold}\nPredicted: {ans}\nCorrect? Accept paraphrases. yes/no.")
    r=oai.chat.completions.create(model="gpt-4o-mini",temperature=0,max_tokens=4,
                                  messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

cc=cw=ab=over=0; N=0; recs=[]
for qi in sorted(test_ids):
    r=full.get(qi)
    if not r: continue
    N+=1
    text=generate(build_prompt(r["question"],r))
    if is_refusal(text):
        ab+=1
        if base_correct_q.get(qi,False): over+=1
        kind="abstain"
    else:
        ans=text.strip().rstrip(".")
        ok=judge(r["question"],r.get("gold_answer",""),ans)
        if ok: cc+=1; kind="commit_correct"
        else:  cw+=1; kind="commit_wrong"
    recs.append({"qi":qi,"kind":kind,"text":text[:200]})
    if N%20==0: print(f"  eval {N}/{len(test_ids)}"); json.dump(recs,open(f"{OUT_DIR}/craft_eval.json","w"))
json.dump(recs,open(f"{OUT_DIR}/craft_eval.json","w"))

def ths(cc,cw,N,base=BASE_ACC): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
print("\n"+"="*54)
print("CRaFT (w/o Flow) on 2WikiMultihopQA — CORRECTED (direct-answer), base 0.440")
print("="*54)
print(f"  cc={cc} cw={cw} ab={ab}")
print(f"  coverage        {(cc+cw)/N:.3f}")
print(f"  selective_acc   {cc/(cc+cw) if (cc+cw) else 0:.3f}")
print(f"  confident_error {cw/N:.3f}")
print(f"  over_abstention {over/max(1,n_base_correct):.3f}")
print(f"  THS             {ths(cc,cw,N):.2f}")
print(f"  (was 8.25 with the buggy reasoning-instruction version)")
json.dump({"cc":cc,"cw":cw,"ab":ab,"N":N,"THS":ths(cc,cw,N),
           "cov":(cc+cw)/N,"sel_acc":cc/(cc+cw) if (cc+cw) else 0,
           "conf_err":cw/N,"over_abst":over/max(1,n_base_correct)},
          open(f"{OUT_DIR}/craft_2wiki_result.json","w"),indent=2)
print("saved -> craft_2wiki_result.json")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  eval 20/200
  eval 40/200
  eval 60/200
  eval 80/200
  eval 100/200
  eval 120/200
  eval 140/200
  eval 160/200
  eval 180/200
  eval 200/200

CRaFT (w/o Flow) on 2WikiMultihopQA — CORRECTED (direct-answer), base 0.440
  cc=43 cw=11 ab=146
  coverage        0.270
  selective_acc   0.796
  confident_error 0.055
  over_abstention 0.489
  THS             17.18
  (was 8.25 with the buggy reasoning-instruction version)
saved -> craft_2wiki_result.json


CRaFT hotpot

In [ ]:
import os, re, json, numpy as np, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)
from sentence_transformers import SentenceTransformer
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_craft_fixed"
ADAPTER    = f"{OUT_DIR}/craft_adapter"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON  = f"{DATA_DIR}/hedge_pre_rl_1000_full.json"   # has final_answers, self_consistency
JUDGE_JSON = f"{DATA_DIR}/rejudged_1000.json"
TEST_Q     = f"{DATA_DIR}/dpo_test_questions.json"
STATE_CACHE= f"{OUT_DIR}/knowledge_state.json"

RETRIEVE_K = 10
BASE_ACC   = 0.515
TAU_MU     = 0.5
RATIO_IDK  = 4
EMBED_ID   = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MAX    = 64

INSTRUCTION = ("Answer the question based on the evidence. Give a brief, direct answer. "
               "If the evidence is insufficient to answer, respond exactly with \"I don't know.\"")
IDK_TEXT = "I don't know."

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
base_judge = {int(x["question_index"]): bool(x["judge_correct"]) for x in json.load(open(JUDGE_JSON))}
test_ids = set(int(x["question_index"]) for x in json.load(open(TEST_Q)))
train_ids = [q for q in full if q not in test_ids]
n_base_correct = sum(base_judge.get(q, False) for q in test_ids)
print(f"train pool: {len(train_ids)}  test: {len(test_ids)}  n_base_correct={n_base_correct}")

ds_tr = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train")
ds_va = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
q2ex = {e["question"]: e for e in ds_tr}
for e in ds_va: q2ex.setdefault(e["question"], e)
def all_sents(ex): return [s.strip() for p in ex["context"]["sentences"] for s in p if s.strip()]
def tk(s): return re.findall(r"\w+", s.lower())
def retrieve(q, ex):
    from rank_bm25 import BM25Okapi
    sents = all_sents(ex)
    if not sents: return []
    bm = BM25Okapi([tk(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(tk(q)))[::-1][:RETRIEVE_K]]
def build_prompt(q, ex):
    ev = "\n".join(f"- {e}" for e in retrieve(q, ex))
    return

train pool: 800  test: 200  n_base_correct=74


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
embedder = SentenceTransformer(EMBED_ID)
import time



def judge_one(q, gold, ans):
    if not ans or not ans.strip(): return False
    p=(f"Question: {q}\nGold: {gold}\nPrediction: {ans}\nCorrect? Accept paraphrases. yes/no.")
    while True:
        try:
          r=oai.chat.completions.create(model="gpt-4o-mini",temperature=0,max_tokens=4,
                                  messages=[{"role":"user","content":p}])
          time.sleep(9)
          return r.choices[0].message.content.strip().lower().startswith("y")

        except RateLimitError:
          print("Rate limit — retrying...")
          time.sleep(10)

def certainty(samples):
    s=[x for x in samples if x and x.strip()]
    if len(s)<2: return 1.0
    E=embedder.encode(s, normalize_embeddings=True); S=E@E.T; n=len(s)
    return float((S.sum()-np.trace(S))/(n*(n-1)))
def majority(samples):
    from collections import Counter
    s=[x for x in samples if x and x.strip()]
    return Counter(s).most_common(1)[0][0] if s else ""


if os.path.exists(STATE_CACHE):
    state={int(k):v for k,v in json.load(open(STATE_CACHE)).items()}
    print(f"resuming: {len(state)} questions already done")
else:
    state={}
for i,qi in enumerate(train_ids):
    if qi in state: continue
    r=full.get(qi)
    if not r or not r.get("final_answers"): continue
    samples=r["final_answers"]; gold=r.get("gold_answer","")
    mu=float(np.mean([judge_one(r["question"],gold,a) for a in samples])) if samples else 0.0
    sigma=float(r["self_consistency"]) if "self_consistency" in r else certainty(samples)
    state[qi]={"mu":mu,"sigma":sigma,"gold":gold,"question":r["question"],
               "majority_answer":majority(samples)}
    json.dump({str(k):v for k,v in state.items()},open(STATE_CACHE,"w"))   # save EVERY question
    if (i+1)%25==0: print(f"  state {i+1}/{len(train_ids)}")
print(f"knowledge state complete: {len(state)} questions")


In [ ]:
van_cand=[(qi,s) for qi,s in state.items() if s["mu"]>=TAU_MU]
idk_cand=[(qi,s) for qi,s in state.items() if s["mu"]< TAU_MU]
van_cand.sort(key=lambda kv:kv[1]["sigma"],reverse=True)
idk_cand.sort(key=lambda kv:kv[1]["sigma"])
N_van=max(1,min(len(van_cand),len(idk_cand)//RATIO_IDK)) if idk_cand else len(van_cand)
N_idk=min(len(idk_cand),N_van*RATIO_IDK)
van_sel,idk_sel=van_cand[:N_van],idk_cand[:N_idk]
print(f"CorCer-RAIT: {len(van_sel)} vanilla + {len(idk_sel)} IdK (1:{RATIO_IDK})")

train_rows=[]
for qi,s in van_sel:
    ex=q2ex.get(s["question"])
    if not ex: continue
    train_rows.append({"prompt":build_prompt(s["question"],ex),"completion":f" {s['majority_answer']}"})
for qi,s in idk_sel:
    ex=q2ex.get(s["question"])
    if not ex: continue
    train_rows.append({"prompt":build_prompt(s["question"],ex),"completion":f" {IDK_TEXT}"})
np.random.shuffle(train_rows)
json.dump(train_rows,open(f"{OUT_DIR}/craft_train_data.json","w"))
print(f"total SFT examples: {len(train_rows)}")
print(f"  sample vanilla: {repr([r for r in train_rows if IDK_TEXT not in r['completion']][0]['completion'])}")
print(f"  sample IdK:     {repr([r for r in train_rows if IDK_TEXT in r['completion']][0]['completion'])}")



In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16)
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb,device_map="auto")
peft_cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",task_type="CAUSAL_LM",
                    target_modules=["q_proj","k_proj","v_proj","o_proj"])
def fmt(ex):
    text=tok.apply_chat_template([{"role":"user","content":ex["prompt"]}],tokenize=False,add_generation_prompt=True)
    return {"text":text+ex["completion"]+tok.eos_token}
sft_ds=Dataset.from_list(train_rows).map(fmt)
cfg=SFTConfig(output_dir=OUT_DIR,per_device_train_batch_size=2,gradient_accumulation_steps=8,
              num_train_epochs=3,learning_rate=1e-5,lr_scheduler_type="cosine",bf16=True,
              logging_steps=10,save_strategy="no",report_to="none")
trainer=SFTTrainer(model=model,args=cfg,train_dataset=sft_ds,peft_config=peft_cfg)
trainer.train(); trainer.save_model(ADAPTER)
print("CRaFT SFT complete.")



In [ ]:
from peft import PeftModel
m=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb,device_map="auto")
m=PeftModel.from_pretrained(m,ADAPTER).eval()

def is_refusal(t):
    tl=t.lower().strip()
    return ("i don't know" in tl) or ("i do not know" in tl)
def generate(prompt):
    inp=tok.apply_chat_template([{"role":"user","content":prompt}],add_generation_prompt=True,
                                return_tensors="pt",return_dict=True).to(m.device)
    with torch.no_grad():
        out=m.generate(**inp,max_new_tokens=GEN_MAX,do_sample=False,pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:],skip_special_tokens=True).strip()
def judge(q,gold,ans):
    if not ans or not ans.strip(): return False
    p=(f"Question: {q}\nGold: {gold}\nPredicted: {ans}\nCorrect? Accept paraphrases. yes/no.")
    r=oai.chat.completions.create(model="gpt-4o-mini",temperature=0,max_tokens=4,
                                  messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

cc=cw=ab=over=0; N=0; recs=[]
for qi in sorted(test_ids):
    r=full.get(qi); ex=q2ex.get(r["question"]) if r else None
    if not r or not ex: continue
    N+=1
    text=generate(build_prompt(r["question"],ex))
    if is_refusal(text):
        ab+=1
        if base_judge.get(qi,False): over+=1
        kind="abstain"
    else:
        ans=text.strip().rstrip(".")
        ok=judge(r["question"],r.get("gold_answer",""),ans)
        if ok: cc+=1; kind="commit_correct"
        else:  cw+=1; kind="commit_wrong"
    recs.append({"qi":qi,"kind":kind,"text":text[:200]})
    if N%20==0: print(f"  eval {N}/{len(test_ids)}"); json.dump(recs,open(f"{OUT_DIR}/craft_eval.json","w"))
json.dump(recs,open(f"{OUT_DIR}/craft_eval.json","w"))

def ths(cc,cw,N,base=BASE_ACC): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
print("\n"+"="*54)
print("CRaFT (w/o Flow) on HotpotQA — CORRECTED (direct-answer), base 0.515")
print("="*54)
print(f"  cc={cc} cw={cw} ab={ab}")
print(f"  coverage        {(cc+cw)/N:.3f}")
print(f"  selective_acc   {cc/(cc+cw) if (cc+cw) else 0:.3f}")
print(f"  confident_error {cw/N:.3f}")
print(f"  over_abstention {over/max(1,n_base_correct):.3f}")
print(f"  THS             {ths(cc,cw,N):.2f}")
print(f"  (was 22.18 with the buggy reasoning-instruction version)")
json.dump({"cc":cc,"cw":cw,"ab":ab,"N":N,"THS":ths(cc,cw,N),
           "cov":(cc+cw)/N,"sel_acc":cc/(cc+cw) if (cc+cw) else 0,
           "conf_err":cw/N,"over_abst":over/max(1,n_base_correct)},
          open(f"{OUT_DIR}/craft_hotpot_result.json","w"),indent=2)
print("saved -> craft_hotpot_result.json")


TruthRL

In [ ]:
!pip install -q "trl==0.15.2" "transformers>=4.48,<4.50" "peft>=0.14" "accelerate>=1.2" \
                "torchao>=0.16.0" bitsandbytes datasets rank-bm25 openai pandas

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   

import re, json, hashlib, random, time
import numpy as np, torch, pandas as pd
from datasets import Dataset, load_dataset
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from openai import OpenAI

random.seed(0); np.random.seed(0); torch.manual_seed(0)


CLOSED_BOOK = False


BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
TAG        = "cb" if CLOSED_BOOK else "ob"
OUT_DIR    = f"{DATA_DIR}/truthrl_grpo_{TAG}"
REWARD_CACHE = f"{OUT_DIR}/reward_cache.json"
os.makedirs(OUT_DIR, exist_ok=True)

RETRIEVE_K = 10
N_TRAIN    = 300
GROUP_SIZE = 8
MAX_NEW    = 64
BASE_ACC   = 0.515
JUDGE_MODEL = "gpt-4o-mini"

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

INSTRUCTION_OB = ("Answer the question using the evidence, reasoning step by step. "
                  "If you can determine the answer, finish with 'Therefore, the answer is X.' "
                  "If the evidence is insufficient or you are not sure, reply exactly "
                  "'I don't know.' It is better to say 'I don't know' than to guess.")
INSTRUCTION_CB = ("Answer the question, reasoning step by step. "
                  "If you know the answer, finish with 'Therefore, the answer is X.' "
                  "If you do not know or are unsure, reply exactly 'I don't know.' "
                  "It is better to say 'I don't know' than to guess.")


In [ ]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train")
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def tkz(s): return re.findall(r"\w+", s.lower())
def all_sents(ex): return [s.strip() for p in ex["context"]["sentences"] for s in p if s.strip()]
def retrieve(ex):
    sents = all_sents(ex)
    if not sents: return []
    bm = BM25Okapi([tkz(s) for s in sents])
    idx = np.argsort(bm.get_scores(tkz(ex["question"])))[::-1][:RETRIEVE_K]
    return [sents[i] for i in idx]

def make_prompt(question, ex=None):
    if CLOSED_BOOK:
        user = f"{INSTRUCTION_CB}\n\nQuestion: {question}"
    else:
        ev = "\n".join(f"- {e}" for e in retrieve(ex))
        user = f"{INSTRUCTION_OB}\n\nEvidence:\n{ev}\n\nQuestion: {question}"
    return tok.apply_chat_template([{"role":"user","content":user}],
                                   tokenize=False, add_generation_prompt=True)

rows = []
for i in random.sample(range(len(ds)), N_TRAIN):
    ex = ds[i]
    rows.append({"prompt": make_prompt(ex["question"], ex),
                 "gold": ex["answer"], "question": ex["question"]})
train_ds = Dataset.from_list(rows)
print(f"built {len(train_ds)} {'CLOSED' if CLOSED_BOOK else 'OPEN'}-book prompts")
print("sample:\n", rows[0]["prompt"][:350])


built 300 OPEN-book prompts
sample:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Answer the question using the evidence, reasoning step by step. If you can determine the answer, finish with 'Therefore, the answer is X.' If the evidence is insufficient or you are not sure, reply exactly 'I don't know.' It is better 


In [ ]:
_reward_cache = {}
if os.path.exists(REWARD_CACHE):
    try: _reward_cache = json.load(open(REWARD_CACHE))
    except Exception: _reward_cache = {}

ABSTAIN_PAT = re.compile(r"\b(i\s*don'?t\s*know|cannot determine|not sure|unsure|"
                         r"insufficient|no answer|unable to answer)\b", re.I)
def is_abstain(text):
    if re.search(r"answer is\s+\S", text, re.I): return False
    return bool(ABSTAIN_PAT.search(text))
def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]

def judge_correct(question, gold, ans):
    key = hashlib.sha256(f"{question}|{gold}|{ans}".encode()).hexdigest()[:20]
    if key in _reward_cache: return _reward_cache[key]
    prompt = (f"Question: {question}\nGround truth: {gold}\nPrediction: {ans}\n"
              "Does the prediction match the ground truth? Accept paraphrases. Reply yes or no.")
    try:
        r = oai.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=4, timeout=20,
                                        messages=[{"role":"user","content":prompt}])
        ok = r.choices[0].message.content.strip().lower().startswith("y")
    except Exception as e:
        print("judge error:", e); ok = False
    _reward_cache[key] = ok
    return ok

_calls = {"n": 0}
def ternary_reward(completions, gold=None, question=None, **kwargs):
    rewards = []
    for comp, g, q in zip(completions, gold, question):
        text = comp if isinstance(comp, str) else comp[0]["content"]
        if is_abstain(text): rewards.append(0.0)
        else: rewards.append(1.0 if judge_correct(q, g, extract_answer(text)) else -1.0)
        _calls["n"] += 1
    if _calls["n"] % 50 < len(completions):
        json.dump(_reward_cache, open(REWARD_CACHE, "w"))
    return rewards

In [ ]:
from trl import GRPOConfig, GRPOTrainer

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                       gradient_checkpointing_kwargs={"use_reentrant": False})
base.enable_input_require_grads()

peft_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules=["q_proj","v_proj"])

cfg = GRPOConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=GROUP_SIZE,   
    gradient_accumulation_steps=1,
    num_generations=GROUP_SIZE,
    max_prompt_length=384,
    max_completion_length=MAX_NEW,
    temperature=1.0,                         
    beta=0.001, learning_rate=1e-6, lr_scheduler_type="constant",
    num_train_epochs=1, logging_steps=1, save_steps=100, bf16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none", log_completions=True,
)

trainer = GRPOTrainer(model=base, reward_funcs=ternary_reward, args=cfg,
                      train_dataset=train_ds, peft_config=peft_cfg)
trainer.train()
trainer.save_model(f"{OUT_DIR}/final_adapter")
json.dump(_reward_cache, open(REWARD_CACHE, "w"))
print(f"TruthRL ({'closed' if CLOSED_BOOK else 'open'}-book) training complete.")


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TruthRL (open-book) training complete.


In [ ]:

log = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["reward","reward_std","kl","loss"] if c in log.columns]
print("=== training signal (last 15 steps) ===")
print(log[cols].dropna().tail(15).to_string())
rs = log["reward_std"].dropna()
print(f"\nmean reward_std: {rs.mean():.3f}  (nonzero => GRPO HAD a learning signal)")
print(f"fraction of steps with reward_std>0: {(rs>0).mean():.2f}")
if rs.mean() > 0.05:
    print(">>> TRAINED: within-group reward variance present. This regime works.")
else:
    print(">>> NO SIGNAL: rollouts were reward-homogeneous in this regime.")

=== training signal (last 15 steps) ===
     reward  reward_std        kl  loss
285   0.750    0.707107  0.000392   0.0
286   0.000    0.000000  0.000878   0.0
287   0.250    1.035098  0.000359   0.0
288   0.250    1.035098  0.000741   0.0
289   1.000    0.000000  0.000127   0.0
290   0.500    0.925820  0.000506   0.0
291   1.000    0.000000  0.000425   0.0
292   0.500    0.925820  0.000736   0.0
293  -0.250    1.035098  0.000517   0.0
294  -1.000    0.000000  0.000674   0.0
295  -1.000    0.000000  0.000480   0.0
296   0.750    0.707107  0.000416   0.0
297   0.625    0.517549  0.000517   0.0
298   0.250    1.035098  0.000419   0.0
299   0.750    0.707107  0.000317   0.0

mean reward_std: 0.515  (nonzero => GRPO HAD a learning signal)
fraction of steps with reward_std>0: 0.61
>>> TRAINED: within-group reward variance present. This regime works.


In [ ]:
FULL_JSON = f"{DATA_DIR}/hedge_pre_rl_1000_full.json"
TEST_JSON = f"{DATA_DIR}/dpo_test_questions.json"
JUDGE_JSON = f"{DATA_DIR}/rejudged_1000.json"
full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
base_judge = {int(x["question_index"]): bool(x["judge_correct"]) for x in json.load(open(JUDGE_JSON))}

dsv = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
q2ex = {e["question"]: e for e in dsv}

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, f"{OUT_DIR}/final_adapter").eval()

def gen(question):
    ex = q2ex.get(question)
    prompt = make_prompt(question, ex)
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=1400).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=96, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

cc=cw=ab=over=0; N=0; recs=[]
for qi in sorted(test_ids):
    r = full.get(qi)
    if not r: continue
    N += 1
    text = gen(r["question"])
    if is_abstain(text):
        ab += 1
        if base_judge.get(qi, False): over += 1
        kind="abstain"
    else:
        ok = judge_correct(r["question"], r.get("gold_answer",""), extract_answer(text))
        if ok: cc+=1; kind="commit_correct"
        else:  cw+=1; kind="commit_wrong"
    recs.append({"qi":qi,"kind":kind,"text":text[:200]})
    if N % 20 == 0:
        print(f"  eval {N}/200"); json.dump(recs, open(f"{OUT_DIR}/eval.json","w"))
json.dump(recs, open(f"{OUT_DIR}/eval.json","w"))

def ths(cc,cw,base=BASE_ACC): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
regime = "closed-book" if CLOSED_BOOK else "open-book (retrieval)"
print("\n" + "="*54)
print(f"TruthRL (GRPO ternary, {regime}) on HotpotQA, base 0.515")
print("="*54)
print(f"  cc={cc} cw={cw} ab={ab}  cov={(cc+cw)/N:.3f}  cErr={cw/N:.3f}  THS={ths(cc,cw):.2f}")
print(f"  (compare: R-Tuning 15.31 | CRaFT 22.18 | P(True) 29.16 | rule 44.63)")
json.dump({"cc":cc,"cw":cw,"ab":ab,"N":N,"THS":ths(cc,cw),"regime":regime},
          open(f"{OUT_DIR}/truthrl_result.json","w"))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  eval 20/200
  eval 40/200
  eval 60/200
  eval 80/200
  eval 100/200
  eval 120/200
  eval 140/200
  eval 160/200
  eval 180/200
  eval 200/200

TruthRL (GRPO ternary, open-book (retrieval)) on HotpotQA, base 0.515
  cc=124 cw=58 ab=18  cov=0.910  cErr=0.290  THS=31.21
  (compare: R-Tuning 15.31 | CRaFT 22.18 | P(True) 29.16 | rule 44.63)


2wiki p(true)

In [ ]:
import os, re, json, numpy as np, torch
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_ptrue_2wiki"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON   = f"{DATA_DIR}/2wiki_full.json"
TEST_JSON   = f"{DATA_DIR}/2wiki_test_questions.json"
TRAIN_JSON  = f"{DATA_DIR}/2wiki_train_questions.json"
FAIR_JSON   = f"{DATA_DIR}/2wiki_base_point_fair.json"

RETRIEVE_K = 10
BASE_ACC   = 0.440
N_VAL      = 150

INSTRUCTION = ("Answer the question by reasoning one step at a time, basing each step on the "
               "evidence. Always finish with 'Therefore, the answer is X.'")

NON_ANSWER = ["cannot determine","cannot be determined","could not determine","does not allow",
    "we cannot","do not have","information provided does not","there is no movie",
    "no movie in the given","cannot definitively","x, where x","x (where",
    "does not provide","not allow us to determine","cannot be determined from",
    "does not contain","cannot be precisely","not confident","cannot verify",
    "unable to determine","insufficient","does not specify","cannot answer"]
def is_non_answer(t): t=str(t).lower(); return any(m in t for m in NON_ANSWER)

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto").eval()

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in set(test_ids)]
val_ids = [q for q in train_ids if q in full][:N_VAL]


fair = json.load(open(FAIR_JSON))
base_correct_q = {int(k): bool(v) for k, v in fair["per_q"].items()}
n_base_correct = sum(base_correct_q.values())
print(f"test={len(test_ids)} val={len(val_ids)} base_correct={n_base_correct}  (expect 88)")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

test=200 val=150 base_correct=88  (expect 88)


In [ ]:
def _tok(s): return re.findall(r"\w+", s.lower())
def all_sents(rec): return [s.strip() for p in rec["context"]["sentences"] for s in p if s and s.strip()]
def top_evidence(rec, q, k=RETRIEVE_K):
    sents = all_sents(rec)
    if not sents: return []
    bm = BM25Okapi([_tok(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(_tok(q)))[::-1][:k]]

def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]

def generate(q, ev_block):
    prompt = f"{INSTRUCTION}\n\nEvidence:\n{ev_block}\n\nQuestion: {q}"
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=220, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# =

In [ ]:
def p_true(q, ans, ev_block):
    prompt = (f"Evidence:\n{ev_block}\n\nQuestion: {q}\nProposed answer: {ans}\n\n"
              "Is the proposed answer True or False? Respond with a single word: True or False.\nAnswer:")
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        logits = lm(**inp).logits[0, -1]
    def tid(w):
        ids = tok(w, add_special_tokens=False).input_ids
        return ids[0] if ids else None
    true_ids = [tid(w) for w in [" True","True"," true","true"] if tid(w) is not None]
    false_ids= [tid(w) for w in [" False","False"," false","false"] if tid(w) is not None]
    probs = torch.softmax(logits.float(), -1)
    pt = float(sum(probs[i] for i in true_ids))
    pf = float(sum(probs[i] for i in false_ids))
    return pt / (pt + pf + 1e-9)


In [ ]:
def judge(q, gold, ans):
    if not ans: return False
    p = (f"Question: {q}\nGold answer: {gold}\nPredicted: {ans}\n"
         "Is the predicted answer correct? Accept paraphrases. Reply yes or no.")
    r = oai.chat.completions.create(model="gpt-4o-mini", temperature=0, max_tokens=4,
                                    messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

def run_split(ids, tag):
    recs = []
    for j, qi in enumerate(ids):
        r = full.get(qi)
        if not r: continue
        ev_block = "\n".join(top_evidence(r, r["question"]))
        text = generate(r["question"], ev_block)
        ans = extract_answer(text)
        pt = p_true(r["question"], ans, ev_block)
        ok = False if is_non_answer(ans) else judge(r["question"], r.get("gold_answer",""), ans)
        recs.append({"qi": qi, "question": r["question"], "gold": r.get("gold_answer",""),
                     "answer": ans, "p_true": pt, "correct": ok, "nonans": is_non_answer(ans)})
        if (j+1) % 25 == 0:
            print(f"  {tag}: {j+1}/{len(ids)}"); json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json","w"))
    json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json","w"))
    return recs

print("=== VALIDATION ==="); val_recs = run_split(val_ids, "val")
print("=== TEST ===");       test_recs = run_split(test_ids, "test")

=== VALIDATION ===
  val: 25/150
  val: 50/150
  val: 75/150
  val: 100/150
  val: 125/150
  val: 150/150
=== TEST ===
  test: 25/200
  test: 50/200
  test: 75/200
  test: 100/200
  test: 125/200
  test: 150/200
  test: 175/200
  test: 200/200


In [ ]:
def ths(cc, cw, N, base=BASE_ACC):
    return ((cc/N)*(1-base) - (cw/N)*base)/(1-base)*100

def evaluate(recs, thr):
    cc=cw=ab=over=0; N=len(recs)
    for x in recs:
        if x["nonans"] or x["p_true"] < thr:
            ab += 1
            if base_correct_q.get(x["qi"], False): over += 1
        else:
            if x["correct"]: cc += 1
            else: cw += 1
    return {"thr":thr,"cc":cc,"cw":cw,"ab":ab,"over":over,"cov":(cc+cw)/N,
            "sel_acc":cc/(cc+cw) if (cc+cw) else 0,"conf_err":cw/N,
            "overAb":over/n_base_correct if n_base_correct else 0,"THS":ths(cc,cw,N)}

vals = sorted(x["p_true"] for x in val_recs)
grid = [vals[int(f*len(vals))] for f in np.arange(0.05, 0.96, 0.05)]
best = max((evaluate(val_recs, t) for t in grid), key=lambda d: d["THS"])
test_result = evaluate(test_recs, best["thr"])

from sklearn.metrics import roc_auc_score
yv = [0 if x["correct"] else 1 for x in test_recs if not x["nonans"]]
sv = [1 - x["p_true"] for x in test_recs if not x["nonans"]]
auroc = roc_auc_score(yv, sv) if len(set(yv)) > 1 else float("nan")

print("\n" + "="*52)
print("P(True) on 2WikiMultihopQA, base 0.440")
print("="*52)
print(f"  best val threshold: P(True) >= {best['thr']:.4f}")
for k,v in test_result.items(): print(f"  {k}: {v}")
print(f"  AUROC (predicting wrong): {auroc:.3f}")
json.dump({"best_thr":best["thr"],"test":test_result,"auroc":auroc},
          open(f"{OUT_DIR}/ptrue_2wiki_result.json","w"))


P(True) on 2WikiMultihopQA, base 0.440
  best val threshold: P(True) >= 0.0001
  thr: 7.484600352033183e-05
  cc: 65
  cw: 41
  ab: 94
  over: 25
  cov: 0.53
  sel_acc: 0.6132075471698113
  conf_err: 0.205
  overAb: 0.2840909090909091
  THS: 16.39285714285715
  AUROC (predicting wrong): 0.648


2wiki verbalized + logprobe

In [ ]:
import os, re, json, numpy as np, torch
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_logprob_verbalized_2wiki"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON   = f"{DATA_DIR}/2wiki_full.json"
TEST_JSON   = f"{DATA_DIR}/2wiki_test_questions.json"
TRAIN_JSON  = f"{DATA_DIR}/2wiki_train_questions.json"
FAIR_JSON   = f"{DATA_DIR}/2wiki_base_point_fair.json"

RETRIEVE_K = 10
BASE_ACC   = 0.440
N_VAL      = 150

INSTRUCTION = ("Answer the question by reasoning one step at a time, basing each step on the "
               "evidence. Always finish with 'Therefore, the answer is X.'")

NON_ANSWER = ["cannot determine","cannot be determined","could not determine","does not allow",
    "we cannot","do not have","information provided does not","there is no movie",
    "no movie in the given","cannot definitively","x, where x","x (where",
    "does not provide","not allow us to determine","cannot be determined from",
    "does not contain","cannot be precisely","not confident","cannot verify",
    "unable to determine","insufficient","does not specify","cannot answer"]
def is_non_answer(t): t=str(t).lower(); return any(m in t for m in NON_ANSWER)

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto").eval()

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in set(test_ids)]
val_ids = [q for q in train_ids if q in full][:N_VAL]

fair = json.load(open(FAIR_JSON))
base_correct_q = {int(k): bool(v) for k, v in fair["per_q"].items()}
n_base_correct = sum(base_correct_q.values())
print(f"test={len(test_ids)} val={len(val_ids)} base_correct={n_base_correct}  (expect 88)")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

test=200 val=150 base_correct=88  (expect 88)


In [ ]:
def _tok(s): return re.findall(r"\w+", s.lower())
def all_sents(rec): return [s.strip() for p in rec["context"]["sentences"] for s in p if s and s.strip()]
def top_evidence(rec, q, k=RETRIEVE_K):
    sents = all_sents(rec)
    if not sents: return []
    bm = BM25Okapi([_tok(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(_tok(q)))[::-1][:k]]

def gen_with_logprobs(q, evb):
    prompt = f"{INSTRUCTION}\n\nEvidence:\n{evb}\n\nQuestion: {q}"
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    out = lm.generate(**inp, max_new_tokens=220, do_sample=False, pad_token_id=tok.pad_token_id,
                      output_scores=True, return_dict_in_generate=True)
    seq = out.sequences[0][inp["input_ids"].shape[1]:]
    lps = [float(torch.log_softmax(s[0].float(), -1)[t]) for s, t in zip(out.scores, seq)]
    ids = seq.tolist()
    text = tok.decode(seq, skip_special_tokens=True).strip()
    return text, lps, ids

def answer_span_logprobs(text, lps, ids):
    """LENGTH-CONFOUND FIX: mean logprob of the answer span only (after 'the answer is')."""
    low = text.lower(); marker = low.rfind("the answer is")
    if marker == -1:
        return lps[-20:] if len(lps) > 20 else lps
    target_char = marker + len("the answer is"); start_tok = None
    for i in range(len(ids)):
        if len(tok.decode(ids[:i+1], skip_special_tokens=True)) >= target_char:
            start_tok = i; break
    if start_tok is None:
        return lps[-20:] if len(lps) > 20 else lps
    return lps[start_tok:] if lps[start_tok:] else lps[-5:]

def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]


In [ ]:
def verbalized_confidence(q, ans, evb):
    prompt = (f"Evidence:\n{evb}\n\nQuestion: {q}\nProposed answer: {ans}\n\n"
              "How confident are you that this answer is correct, given the evidence? "
              "Respond with ONLY a number from 0 to 100 (0 = certainly wrong, 100 = certainly correct).")
    inp = tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=6, do_sample=False, pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\d{1,3}", txt)
    if not m: return 0.5
    return min(100, max(0, int(m.group(0)))) / 100.0


In [ ]:
def judge(q, gold, ans):
    if not ans: return False
    p = (f"Question: {q}\nGold answer: {gold}\nPredicted: {ans}\n"
         "Is the predicted answer correct? Accept paraphrases. Reply yes or no.")
    r = oai.chat.completions.create(model="gpt-4o-mini", temperature=0, max_tokens=4,
                                    messages=[{"role":"user","content":p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

def run_split(ids, tag):
    recs = []
    for j, qi in enumerate(ids):
        r = full.get(qi)
        if not r: continue
        evb = "\n".join(top_evidence(r, r["question"]))
        text, lps, tids = gen_with_logprobs(r["question"], evb)
        ans = extract_answer(text)
        span = answer_span_logprobs(text, lps, tids)
        logprob_score = float(np.mean(span)) if span else -1e9
        vconf = verbalized_confidence(r["question"], ans, evb)
        nonans = is_non_answer(ans)
        ok = False if nonans else judge(r["question"], r.get("gold_answer",""), ans)
        recs.append({"qi": qi, "question": r["question"], "gold": r.get("gold_answer",""),
                     "answer": ans, "logprob": logprob_score, "verbalized": vconf,
                     "correct": ok, "nonans": nonans})
        if (j+1) % 25 == 0:
            print(f"  {tag}: {j+1}/{len(ids)}"); json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json","w"))
    json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json","w"))
    return recs

print("=== VALIDATION ==="); val_recs = run_split(val_ids, "val")
print("=== TEST ===");       test_recs = run_split(test_ids, "test")


=== VALIDATION ===
  val: 25/150
  val: 50/150
  val: 75/150
  val: 100/150
  val: 125/150
  val: 150/150
=== TEST ===
  test: 25/200
  test: 50/200
  test: 75/200
  test: 100/200
  test: 125/200
  test: 150/200
  test: 175/200
  test: 200/200


In [ ]:
from sklearn.metrics import roc_auc_score
def ths(cc, cw, N, base=BASE_ACC): return ((cc/N)*(1-base) - (cw/N)*base)/(1-base)*100

def evaluate(recs, signal, thr):
    cc=cw=ab=over=0; N=len(recs)
    for x in recs:
        if x["nonans"] or x[signal] < thr:
            ab += 1
            if base_correct_q.get(x["qi"], False): over += 1
        else:
            if x["correct"]: cc += 1
            else: cw += 1
    return {"thr":thr,"cc":cc,"cw":cw,"ab":ab,"over":over,"cov":(cc+cw)/N,
            "sel_acc":cc/(cc+cw) if (cc+cw) else 0,"conf_err":cw/N,
            "overAb":over/n_base_correct if n_base_correct else 0,"THS":ths(cc,cw,N)}

def auroc_for(signal):
    y = [0 if x["correct"] else 1 for x in test_recs if not x["nonans"]]
    if signal == "verbalized":
        s = [1 - x["verbalized"] for x in test_recs if not x["nonans"]]
    else:
        s = [-x["logprob"] for x in test_recs if not x["nonans"]]
    return roc_auc_score(y, s) if len(set(y)) > 1 else float("nan")

def tune_and_eval(signal, name):
    vals = sorted(x[signal] for x in val_recs)
    grid = [vals[int(f*len(vals))] for f in np.arange(0.05, 0.96, 0.05)]
    best = max((evaluate(val_recs, signal, t) for t in grid), key=lambda d: d["THS"])
    tr = evaluate(test_recs, signal, best["thr"])
    au = auroc_for(signal)
    print(f"\n=== {name} on 2wiki (base 0.440) ===")
    print(f"  best val threshold: {signal} >= {best['thr']:.4f}")
    for k,v in tr.items(): print(f"  {k}: {v}")
    print(f"  AUROC (predicting wrong): {au:.3f}")
    return {"best_thr":best["thr"],"test":tr,"auroc":au}

res_logprob = tune_and_eval("logprob", "Logprob threshold (Ren/Jurayj)")
res_verbal  = tune_and_eval("verbalized", "Verbalized confidence (Tian et al.)")

json.dump({"logprob":res_logprob, "verbalized":res_verbal},
          open(f"{OUT_DIR}/logprob_verbalized_2wiki_results.json","w"))
print("\n" + "="*54)
print(f"  Logprob threshold:      THS {res_logprob['test']['THS']:.2f}  AUROC {res_logprob['auroc']:.3f}")
print(f"  Verbalized confidence:  THS {res_verbal['test']['THS']:.2f}  AUROC {res_verbal['auroc']:.3f}")
print(f"  (2wiki compare: P(True) THS 16.39 AUROC 0.648)")
print("="*54)


=== Logprob threshold (Ren/Jurayj) on 2wiki (base 0.440) ===
  best val threshold: logprob >= -0.1650
  thr: -0.16504331957428803
  cc: 59
  cw: 39
  ab: 102
  over: 27
  cov: 0.49
  sel_acc: 0.6020408163265306
  conf_err: 0.195
  overAb: 0.3068181818181818
  THS: 14.178571428571429
  AUROC (predicting wrong): 0.590

=== Verbalized confidence (Tian et al.) on 2wiki (base 0.440) ===
  best val threshold: verbalized >= 0.6000
  thr: 0.6
  cc: 73
  cw: 43
  ab: 84
  over: 18
  cov: 0.58
  sel_acc: 0.6293103448275862
  conf_err: 0.215
  overAb: 0.20454545454545456
  THS: 19.607142857142858
  AUROC (predicting wrong): 0.616

  Logprob threshold:      THS 14.18  AUROC 0.590
  Verbalized confidence:  THS 19.61  AUROC 0.616
  (2wiki compare: P(True) THS 16.39 AUROC 0.648)


2wiki TruthRL

In [ ]:
!pip install -q "trl==0.15.2" "transformers>=4.48,<4.50" "peft>=0.14" "accelerate>=1.2" \
                "torchao>=0.16.0" bitsandbytes datasets rank-bm25 openai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re, json, hashlib, random
import numpy as np, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from openai import OpenAI

random.seed(0); np.random.seed(0); torch.manual_seed(0)

# open-book gives GRPO its reward variance (HotpotQA lesson). Keep True.
CLOSED_BOOK = False

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
TAG        = "cb" if CLOSED_BOOK else "ob"
OUT_DIR    = f"{DATA_DIR}/truthrl_2wiki_{TAG}"
REWARD_CACHE = f"{OUT_DIR}/reward_cache.json"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON  = f"{DATA_DIR}/2wiki_full.json"
TEST_JSON  = f"{DATA_DIR}/2wiki_test_questions.json"
TRAIN_JSON = f"{DATA_DIR}/2wiki_train_questions.json"
FAIR_JSON  = f"{DATA_DIR}/2wiki_base_point_fair.json"

RETRIEVE_K = 10
N_TRAIN    = 300
GROUP_SIZE = 8
MAX_NEW    = 64
BASE_ACC   = 0.440
JUDGE_MODEL = "gpt-4o-mini"

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

INSTRUCTION_OB = ("Answer the question using the evidence, reasoning step by step. "
                  "If you can determine the answer, finish with 'Therefore, the answer is X.' "
                  "If the evidence is insufficient or you are not sure, reply exactly "
                  "'I don't know.' It is better to say 'I don't know' than to guess.")
INSTRUCTION_CB = ("Answer the question, reasoning step by step. "
                  "If you know the answer, finish with 'Therefore, the answer is X.' "
                  "If you do not know or are unsure, reply exactly 'I don't know.' "
                  "It is better to say 'I don't know' than to guess.")


In [ ]:
from rank_bm25 import BM25Okapi
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = set(int(x["question_index"]) for x in json.load(open(TEST_JSON)))
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in test_ids]
train_ids = [q for q in train_ids if q in full and q not in test_ids]

def _tok(s): return re.findall(r"\w+", s.lower())
def all_sents(rec): return [s.strip() for p in rec["context"]["sentences"] for s in p if s and s.strip()]
def retrieve(rec):
    sents = all_sents(rec)
    if not sents: return []
    bm = BM25Okapi([_tok(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(_tok(rec["question"])))[::-1][:RETRIEVE_K]]

def make_prompt(rec):
    q = rec["question"]
    if CLOSED_BOOK:
        user = f"{INSTRUCTION_CB}\n\nQuestion: {q}"
    else:
        ev = "\n".join(f"- {e}" for e in retrieve(rec))
        user = f"{INSTRUCTION_OB}\n\nEvidence:\n{ev}\n\nQuestion: {q}"
    return tok.apply_chat_template([{"role":"user","content":user}],
                                   tokenize=False, add_generation_prompt=True)

rng = random.Random(0)
chosen = rng.sample(train_ids, min(N_TRAIN, len(train_ids)))
rows = [{"prompt": make_prompt(full[qi]), "gold": full[qi].get("gold_answer",""),
         "question": full[qi]["question"]} for qi in chosen]
train_ds = Dataset.from_list(rows)
print(f"built {len(train_ds)} {'CLOSED' if CLOSED_BOOK else 'OPEN'}-book 2wiki prompts")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

built 300 OPEN-book 2wiki prompts


In [ ]:
_reward_cache = {}
if os.path.exists(REWARD_CACHE):
    try: _reward_cache = json.load(open(REWARD_CACHE))
    except Exception: _reward_cache = {}

ABSTAIN_PAT = re.compile(r"\b(i\s*don'?t\s*know|cannot determine|not sure|unsure|"
                         r"insufficient|no answer|unable to answer)\b", re.I)
def is_abstain(text):
    if re.search(r"answer is\s+\S", text, re.I): return False
    return bool(ABSTAIN_PAT.search(text))
def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:120]

def judge_correct(question, gold, ans):
    key = hashlib.sha256(f"{question}|{gold}|{ans}".encode()).hexdigest()[:20]
    if key in _reward_cache: return _reward_cache[key]
    prompt = (f"Question: {question}\nGround truth: {gold}\nPrediction: {ans}\n"
              "Does the prediction match the ground truth? Accept paraphrases. Reply yes or no.")
    try:
        r = oai.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=4, timeout=20,
                                        messages=[{"role":"user","content":prompt}])
        ok = r.choices[0].message.content.strip().lower().startswith("y")
    except Exception as e:
        print("judge error:", e); ok = False
    _reward_cache[key] = ok
    return ok

_calls = {"n": 0}
def ternary_reward(completions, gold=None, question=None, **kwargs):
    rewards = []
    for comp, g, q in zip(completions, gold, question):
        text = comp if isinstance(comp, str) else comp[0]["content"]
        if is_abstain(text): rewards.append(0.0)
        else: rewards.append(1.0 if judge_correct(q, g, extract_answer(text)) else -1.0)
        _calls["n"] += 1
    if _calls["n"] % 50 < len(completions):
        json.dump(_reward_cache, open(REWARD_CACHE, "w"))
    return rewards


In [ ]:
from trl import GRPOConfig, GRPOTrainer

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                       gradient_checkpointing_kwargs={"use_reentrant": False})
base.enable_input_require_grads()

peft_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules=["q_proj","v_proj"])

cfg = GRPOConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=GROUP_SIZE,
    gradient_accumulation_steps=1,
    num_generations=GROUP_SIZE,
    max_prompt_length=384,
    max_completion_length=MAX_NEW,
    temperature=1.0, beta=0.001, learning_rate=1e-6, lr_scheduler_type="constant",
    num_train_epochs=1, logging_steps=1, save_steps=100, bf16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none", log_completions=True,
)

trainer = GRPOTrainer(model=base, reward_funcs=ternary_reward, args=cfg,
                      train_dataset=train_ds, peft_config=peft_cfg)
trainer.train()
trainer.save_model(f"{OUT_DIR}/final_adapter")
json.dump(_reward_cache, open(REWARD_CACHE, "w"))
print(f"TruthRL 2wiki ({'closed' if CLOSED_BOOK else 'open'}-book) training complete.")



config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TruthRL 2wiki (open-book) training complete.


In [ ]:
log = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["reward","reward_std","kl","loss"] if c in log.columns]
print("=== training signal (last 15 steps) ===")
print(log[cols].dropna().tail(15).to_string())
rs = log["reward_std"].dropna()
print(f"\nmean reward_std: {rs.mean():.3f}  (nonzero => GRPO had a signal)")
print(f"fraction steps reward_std>0: {(rs>0).mean():.2f}")
print(">>> TRAINED" if rs.mean() > 0.05 else ">>> NO SIGNAL (rollouts homogeneous)")


=== training signal (last 15 steps) ===
     reward  reward_std        kl  loss
285  -0.750    0.707107  0.000471   0.0
286  -0.250    1.035098  0.000500   0.0
287  -1.000    0.000000  0.000574   0.0
288   0.125    0.353553  0.000725   0.0
289  -0.625    0.744024  0.000640   0.0
290  -0.500    0.925820  0.000400   0.0
291   0.000    0.000000  0.000525   0.0
292   0.125    0.991031  0.000448   0.0
293  -0.750    0.707107  0.000535   0.0
294   1.000    0.000000  0.000394   0.0
295   0.500    0.925820  0.000253   0.0
296  -0.750    0.707107  0.000465   0.0
297  -0.500    0.534522  0.000430   0.0
298   1.000    0.000000  0.000278   0.0
299   0.000    0.000000  0.000455   0.0

mean reward_std: 0.564  (nonzero => GRPO had a signal)
fraction steps reward_std>0: 0.69
>>> TRAINED


In [ ]:
fair = json.load(open(FAIR_JSON))
base_correct_q = {int(k): bool(v) for k, v in fair["per_q"].items()}
n_base_correct = sum(base_correct_q.values())
print(f"base_correct={n_base_correct} (expect 88)")

test_id_list = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, f"{OUT_DIR}/final_adapter").eval()

def gen(rec):
    prompt = make_prompt(rec)
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=1400).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=96, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

cc=cw=ab=over=0; N=0; recs=[]
for qi in sorted(test_id_list):
    r = full.get(qi)
    if not r: continue
    N += 1
    text = gen(r)
    if is_abstain(text):
        ab += 1
        if base_correct_q.get(qi, False): over += 1
        kind="abstain"
    else:
        ok = judge_correct(r["question"], r.get("gold_answer",""), extract_answer(text))
        if ok: cc+=1; kind="commit_correct"
        else:  cw+=1; kind="commit_wrong"
    recs.append({"qi":qi,"kind":kind,"text":text[:200]})
    if N % 20 == 0:
        print(f"  eval {N}/200"); json.dump(recs, open(f"{OUT_DIR}/eval.json","w"))
json.dump(recs, open(f"{OUT_DIR}/eval.json","w"))

def ths(cc,cw,base=BASE_ACC): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
regime = "closed-book" if CLOSED_BOOK else "open-book (retrieval)"
print("\n" + "="*54)
print(f"TruthRL (GRPO ternary, {regime}) on 2WikiMultihopQA, base 0.440")
print("="*54)
print(f"  cc={cc} cw={cw} ab={ab}  cov={(cc+cw)/N:.3f}  cErr={cw/N:.3f}  overAb={over/max(1,n_base_correct):.3f}  THS={ths(cc,cw):.2f}")
print(f"  (2wiki compare: R-Tuning 12.64 | logprob 14.18 | P(True) 16.39 | verbalized 19.61)")
json.dump({"cc":cc,"cw":cw,"ab":ab,"N":N,"THS":ths(cc,cw),"regime":regime},
          open(f"{OUT_DIR}/truthrl_2wiki_result.json","w"))


base_correct=88 (expect 88)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  eval 20/200
  eval 40/200
  eval 60/200
  eval 80/200
  eval 100/200
  eval 120/200
  eval 140/200
  eval 160/200
  eval 180/200
  eval 200/200

TruthRL (GRPO ternary, open-book (retrieval)) on 2WikiMultihopQA, base 0.440
  cc=98 cw=46 ab=56  cov=0.720  cErr=0.230  overAb=0.136  THS=30.93
  (2wiki compare: R-Tuning 12.64 | logprob 14.18 | P(True) 16.39 | verbalized 19.61)


SE

In [ ]:
import os, re, json, math, numpy as np, torch
from rank_bm25 import BM25Okapi
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          AutoModelForSequenceClassification)
from openai import OpenAI

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
NLI_MODEL  = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
DATA_DIR   = "/content/drive/MyDrive/hedge_run"
OUT_DIR    = f"{DATA_DIR}/baseline_semantic_entropy_2wiki"
os.makedirs(OUT_DIR, exist_ok=True)

FULL_JSON  = f"{DATA_DIR}/2wiki_full.json"
TEST_JSON  = f"{DATA_DIR}/2wiki_test_questions.json"
TRAIN_JSON = f"{DATA_DIR}/2wiki_train_questions.json"
FAIR_JSON  = f"{DATA_DIR}/2wiki_base_point_fair.json"

RETRIEVE_K = 10
MAX_TOK    = 220
BASE_ACC   = 0.440
K_SAMPLES  = 10
SAMPLE_TEMP = 1.0
ENTAIL_THR = 0.50
N_VAL      = 150


NON_ANSWER = ["cannot determine","cannot be determined","could not determine","does not allow",
    "we cannot","do not have","information provided does not","there is no movie",
    "no movie in the given","cannot definitively","x, where x","x (where",
    "does not provide","not allow us to determine","cannot be determined from",
    "does not contain","cannot be precisely","not confident","cannot verify",
    "unable to determine","insufficient","does not specify","cannot answer"]
def is_nonanswer(a): a=str(a).lower(); return any(m in a for m in NON_ANSWER)

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
oai = OpenAI(api_key=OPENAI_API_KEY)

INSTRUCTION = ("Answer the question by reasoning one step at a time, basing each step on the "
               "evidence. Always finish with 'Therefore, the answer is X.'")


In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
lm = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                          device_map="auto").eval()

nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).eval()
if torch.cuda.is_available(): nli = nli.to("cuda")
id2 = nli.config.id2label
ENT = [i for i, v in id2.items() if "entail" in v.lower()][0]

def nli_entail(premise, hyp) -> float:
    t = nli_tok(premise, hyp, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available(): t = {k: v.to("cuda") for k, v in t.items()}
    with torch.no_grad():
        p = torch.softmax(nli(**t).logits, -1)[0]
    return float(p[ENT])

full = {int(r["question_index"]): r for r in json.load(open(FULL_JSON))}
test_ids = [int(x["question_index"]) for x in json.load(open(TEST_JSON))]
try:
    train_ids = [int(x["question_index"]) for x in json.load(open(TRAIN_JSON))]
except Exception:
    train_ids = [q for q in full if q not in set(test_ids)]
val_ids = [q for q in train_ids if q in full][:N_VAL]

fair = json.load(open(FAIR_JSON))
base_correct_q = {int(k): bool(v) for k, v in fair["per_q"].items()}
n_base_correct = sum(base_correct_q.values())
print(f"test={len(test_ids)} val={len(val_ids)} base_correct={n_base_correct} (expect 88)")

def tk(s): return re.findall(r"\w+", s.lower())
def all_sents(rec): return [s.strip() for p in rec["context"]["sentences"] for s in p if s and s.strip()]
def retrieve(q, rec):
    sents = all_sents(rec)
    if not sents: return []
    bm = BM25Okapi([tk(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(tk(q)))[::-1][:RETRIEVE_K]]


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

test=200 val=150 base_correct=88 (expect 88)


In [ ]:
def build_prompt(question, evidence):
    ev = "\n".join(f"- {e}" for e in evidence)
    return f"{INSTRUCTION}\n\nEvidence:\n{ev}\n\nQuestion: {question}"

def extract_answer(text):
    m = re.search(r"answer is[:\s]+(.*)", text, re.I)
    return (m.group(1).strip().rstrip(".") if m else text.strip().split("\n")[-1])[:100]

def sample_answers(question, evidence, k=K_SAMPLES):
    msg = build_prompt(question, evidence)
    inp = tok.apply_chat_template([{"role": "user", "content": msg}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(lm.device)
    answers = []
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=MAX_TOK, do_sample=True,
                          temperature=SAMPLE_TEMP, top_p=0.9, num_return_sequences=k,
                          pad_token_id=tok.pad_token_id)
    plen = inp["input_ids"].shape[1]
    for seq in out:
        txt = tok.decode(seq[plen:], skip_special_tokens=True).strip()
        answers.append(extract_answer(txt))
    return answers


In [ ]:
def same_meaning(a, b) -> bool:
    if a.strip().lower() == b.strip().lower():
        return True
    return (nli_entail(a, b) > ENTAIL_THR) and (nli_entail(b, a) > ENTAIL_THR)

def semantic_entropy(answers):
    answers = [a for a in answers if a and a.strip() and not is_nonanswer(a)]
    if not answers:
        return None, 0, ""
    clusters, assign = [], []
    for a in answers:
        placed = False
        for ci, rep in enumerate(clusters):
            if same_meaning(a, rep):
                assign.append(ci); placed = True; break
        if not placed:
            clusters.append(a); assign.append(len(clusters) - 1)
    counts = np.bincount(assign, minlength=len(clusters)).astype(float)
    probs = counts / counts.sum()
    ent = float(-(probs * np.log(probs + 1e-12)).sum())
    maj_ci = int(np.argmax(counts))
    return ent, len(clusters), clusters[maj_ci]

In [ ]:
def run_split(ids, tag):
    path = f"{OUT_DIR}/{tag}_records.json"
    recs = json.load(open(path)) if os.path.exists(path) else []
    done = {x["qi"] for x in recs}
    for j, qi in enumerate(ids):
        if qi in done: continue
        r = full.get(qi)
        if not r: continue
        ev = retrieve(r["question"], r)
        samples = sample_answers(r["question"], ev)
        ent, ncl, maj = semantic_entropy(samples)
        recs.append({"qi": qi, "question": r["question"], "gold": r.get("gold_answer",""),
                     "samples": samples, "entropy": ent, "n_clusters": ncl,
                     "majority_answer": maj})
        if (j + 1) % 20 == 0:
            print(f"  {tag}: {j+1}/{len(ids)}"); json.dump(recs, open(path, "w"))
    json.dump(recs, open(path, "w"))
    return recs

print("=== sampling VALIDATION ===")
val_recs = run_split(val_ids, "val")
print("=== sampling TEST ===")
test_recs = run_split(test_ids, "test")

=== sampling VALIDATION ===
  val: 20/150
  val: 40/150
  val: 60/150
  val: 80/150
  val: 100/150
  val: 120/150
  val: 140/150
=== sampling TEST ===
  test: 20/200
  test: 40/200
  test: 60/200
  test: 80/200
  test: 100/200
  test: 120/200
  test: 140/200
  test: 160/200
  test: 180/200
  test: 200/200


In [ ]:
def judge(question, gold, ans):
    if not ans or is_nonanswer(ans): return False
    p = (f"Question: {question}\nGold answer: {gold}\nPredicted: {ans}\n"
         "Is the predicted answer correct? Accept paraphrases/equivalent. Reply yes or no.")
    r = oai.chat.completions.create(model="gpt-4o-mini", temperature=0, max_tokens=4,
                                    messages=[{"role": "user", "content": p}])
    return r.choices[0].message.content.strip().lower().startswith("y")

for recs, tag in ((val_recs,"val"), (test_recs,"test")):
    for x in recs:
        if "correct" not in x:
            x["correct"] = judge(x["question"], x["gold"], x["majority_answer"])
    json.dump(recs, open(f"{OUT_DIR}/{tag}_records.json", "w"))

N_TEST = len(test_recs)
def ths(cc, cw, N, base=BASE_ACC):
    return ((cc/N)*(1-base) - (cw/N)*base)/(1-base)*100

def evaluate(recs, thr):
    cc = cw = ab = over = 0; N = len(recs)
    for x in recs:
        e = x["entropy"]
        if e is None or e > thr:
            ab += 1
            if base_correct_q.get(x["qi"], False): over += 1
        else:
            if x["correct"]: cc += 1
            else: cw += 1
    n_bc = sum(base_correct_q.get(x["qi"], False) for x in recs)
    return {"thr": thr, "cc": cc, "cw": cw, "ab": ab, "over": over,
            "cov": (cc+cw)/N, "sel_acc": cc/(cc+cw) if (cc+cw) else 0,
            "conf_err": cw/N, "over_abst": over/max(1,n_bc), "THS": ths(cc, cw, N)}

ent_vals = [x["entropy"] for x in val_recs if x["entropy"] is not None]
lo, hi = (min(ent_vals), max(ent_vals)) if ent_vals else (0.0, 2.5)
grid = [round(t, 3) for t in np.linspace(lo, hi, 25)]
best = max((evaluate(val_recs, t) for t in grid), key=lambda d: d["THS"])
best_thr = best["thr"]
print(f"\nbest val entropy threshold: abstain if entropy > {best_thr} (val THS {best['THS']:.2f})")

test_result = evaluate(test_recs, best_thr)
print("\n=== Semantic Entropy on 2WikiMultihopQA TEST (base 0.440) ===")
for k, v in test_result.items():
    print(f"  {k}: {v}")
print(f"\n  2wiki compare: R-Tuning 14.39 | P(True) 16.39 | verbalized 19.61 | TruthRL 30.93")

# save per-q entropy for rule+SE on 2wiki
se_by_q = {str(x["qi"]): x["entropy"] for x in test_recs}
json.dump(se_by_q, open(f"{DATA_DIR}/se_2wiki_by_q.json", "w"))
json.dump({"best_thr": best_thr, "test": test_result, "K_SAMPLES": K_SAMPLES},
          open(f"{OUT_DIR}/semantic_entropy_2wiki_result.json", "w"))
print("\nsaved -> semantic_entropy_2wiki_result.json + se_2wiki_by_q.json (for rule+SE)")


best val entropy threshold: abstain if entropy > 1.919 (val THS 22.48)

=== Semantic Entropy on 2WikiMultihopQA TEST (base 0.440) ===
  thr: 1.919
  cc: 72
  cw: 45
  ab: 83
  over: 24
  cov: 0.585
  sel_acc: 0.6153846153846154
  conf_err: 0.225
  over_abst: 0.2727272727272727
  THS: 18.32142857142857

  2wiki compare: R-Tuning 14.39 | P(True) 16.39 | verbalized 19.61 | TruthRL 30.93

saved -> semantic_entropy_2wiki_result.json + se_2wiki_by_q.json (for rule+SE)


In [ ]:

import os, json, glob, numpy as np
DATA_DIR="/content/drive/MyDrive/hedge_run"
fair=json.load(open(f"{DATA_DIR}/base_point_fair.json"))
BASE=fair["base_accuracy"]
base_correct_q={int(k):bool(v) for k,v in fair["per_q"].items()}
test_ids=[int(x["question_index"]) for x in json.load(open(f"{DATA_DIR}/dpo_test_questions.json"))]
n_base_correct=sum(base_correct_q.get(q,False) for q in test_ids)
assert abs(BASE-0.48)<1e-6 and n_base_correct==96, "clean base mismatch"
print(f"clean base {BASE:.3f}  n_base_correct={n_base_correct}\n")

def ths(cc,cw,N,base=BASE): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
def load(p): return json.load(open(p)) if os.path.exists(p) else None

def score_threshold(val_recs, test_recs, sig, grid, abstain_if_below=True):
    """abstain_if_below: abstain when signal<thr (P(True)/logprob/verbalized).
       else abstain when signal>thr (entropy)."""
    def ev(recs, thr):
        cc=cw=0; ab_q=[]
        for x in recs:
            s=x.get(sig)
            if s is None: ab_q.append(x["qi"]); continue
            commit = (s>=thr) if abstain_if_below else (s<=thr)
            if commit:
                if x["correct"]: cc+=1
                else: cw+=1
            else: ab_q.append(x["qi"])
        over=sum(base_correct_q.get(q,False) for q in ab_q); N=len(recs)
        return {"thr":thr,"cc":cc,"cw":cw,"ab":len(ab_q),
                "cov":(cc+cw)/N,"sel_acc":cc/(cc+cw) if cc+cw else 0,
                "conf_err":cw/N,"over_abst":over/max(1,n_base_correct),"THS":ths(cc,cw,N)}
    best=max((ev(val_recs,t) for t in grid), key=lambda d:d["THS"])
    r=ev(test_recs,best["thr"]); r["tuned_thr"]=best["thr"]; return r

def recount(recs):
    cc=sum(r.get("kind")=="commit_correct" for r in recs)
    cw=sum(r.get("kind")=="commit_wrong" for r in recs)
    ab=[int(r.get("qi",r.get("question_index"))) for r in recs if r.get("kind")=="abstain"]
    over=sum(base_correct_q.get(q,False) for q in ab); N=len(recs)
    return {"cc":cc,"cw":cw,"ab":len(ab),"cov":(cc+cw)/N,
            "sel_acc":cc/(cc+cw) if cc+cw else 0,"conf_err":cw/N,
            "over_abst":over/max(1,n_base_correct),"THS":ths(cc,cw,N),"tuned_thr":None}

R={}

d=f"{DATA_DIR}/baseline_ptrue"; val,test=load(f"{d}/val_records.json"),load(f"{d}/test_records.json")
if val and test:
    for r in val+test: r["qi"]=int(r.get("qi",r.get("question_index")))
    grid=[round(t,2) for t in np.arange(0.30,0.96,0.02)]
    R["P(True)"]=score_threshold(val,test,"p_true",grid,abstain_if_below=True)

d=f"{DATA_DIR}/baseline_logprob_verbalized"; val,test=load(f"{d}/val_records.json"),load(f"{d}/test_records.json")
if val and test:
    for r in val+test: r["qi"]=int(r.get("qi",r.get("question_index")))
    for sig,name,mode in [("logprob","logprob","quantile"),("verbalized","verbalized","distinct")]:
        raw=[x[sig] for x in val if x.get(sig) is not None]
        if len(set(raw))<2: continue
        if mode=="distinct":

            vals=sorted(set(raw)); grid=[v for v in vals if v>vals[0]]
        else:

            vals=sorted(raw)
            grid=sorted(set(vals[min(int(f*len(vals)),len(vals)-1)] for f in np.arange(0.05,0.96,0.05)))
            grid=[g for g in grid if g>vals[0]]
        R[name]=score_threshold(val,test,sig,grid,abstain_if_below=True)

d=f"{DATA_DIR}/baseline_semantic_entropy"; val,test=load(f"{d}/val_records.json"),load(f"{d}/test_records.json")
if val and test:
    for r in val+test: r["qi"]=int(r["qi"])
    vals=sorted(x["entropy"] for x in val if x.get("entropy") is not None)
    grid=[vals[min(int(f*len(vals)),len(vals)-1)] for f in np.arange(0.05,0.96,0.05)]
    R["SemEntropy"]=score_threshold(val,test,"entropy",grid,abstain_if_below=False)

recs=load(f"{DATA_DIR}/baseline_craft_fixed/craft_eval.json")
if recs: R["CRaFT"]=recount(recs)


for tag in ["ob","cb"]:
    recs=load(f"{DATA_DIR}/truthrl_grpo_{tag}/eval.json")
    if recs and any(r.get("kind") for r in recs): R[f"TruthRL-{tag}"]=recount(recs)


def find_rtuning():
    pats=["*rtuning*eval*.json","*rtuning*result*.json","*rtuning*outcome*.json",
          "eval_rtuning*.json","*rtuning*progress*.json","*rtuning*hotpot*.json"]
    cands=[]
    for p in pats: cands+=glob.glob(f"{DATA_DIR}/{p}")
    for c in sorted(set(cands)):
        try: d=json.load(open(c))
        except: continue
        recs = d if isinstance(d,list) else (list(d.values()) if isinstance(d,dict) else None)
        if not recs or not isinstance(recs[0],dict): continue

        if "2wiki" in c.lower(): continue

        if any("kind" in r for r in recs) or any("correct" in r for r in recs):
            return c, recs
    return None,None
c,recs=find_rtuning()
if recs:
    print(f"R-Tuning records: {os.path.basename(c)} (n={len(recs)}, keys={list(recs[0].keys())})")
    kinds=set(r.get("kind") for r in recs)
    if kinds & {"stop","commit"}:

        cc=sum(1 for r in recs if r.get("kind")=="commit" and r.get("correct"))
        cw=sum(1 for r in recs if r.get("kind")=="commit" and not r.get("correct"))

        ab=[test_ids[i] for i,r in enumerate(recs) if r.get("kind")=="stop" and i<len(test_ids)]
        over=sum(base_correct_q.get(q,False) for q in ab); N=len(recs)
        R["R-Tuning"]={"cc":cc,"cw":cw,"ab":len(ab),"cov":(cc+cw)/N,
                       "sel_acc":cc/(cc+cw) if cc+cw else 0,"conf_err":cw/N,
                       "over_abst":over/max(1,n_base_correct),"THS":ths(cc,cw,N),"tuned_thr":None}
    elif any(r.get("kind") in ("commit_correct","commit_wrong","abstain") for r in recs):
        R["R-Tuning"]=recount(recs)
    else:

        cc=cw=0; ab=[]
        for r in recs:
            qi=int(r.get("qi",r.get("question_index",-1)))
            refused = r.get("abstain") or r.get("is_refusal") or (str(r.get("answer","")).strip().lower() in ("i am unsure.","i don't know.","unsure"))
            if refused: ab.append(qi)
            elif r.get("correct"): cc+=1
            else: cw+=1
        over=sum(base_correct_q.get(q,False) for q in ab); N=len(recs)
        R["R-Tuning"]={"cc":cc,"cw":cw,"ab":len(ab),"cov":(cc+cw)/N,
                       "sel_acc":cc/(cc+cw) if cc+cw else 0,"conf_err":cw/N,
                       "over_abst":over/max(1,n_base_correct),"THS":ths(cc,cw,N),"tuned_thr":None}
else:
    print("R-Tuning HotpotQA records NOT found — likely in final_experiment.ipynb output dir.\n"
          "  (paste the filename and I'll wire it in.)")

print("\n"+"="*84)
print(f"HotpotQA — UNIFORM RECOMPUTE v2 @ clean base {BASE:.3f}  (n_base_correct={n_base_correct})")
print("="*84)
print(f"{'method':<16}{'cov':>7}{'selAcc':>8}{'cErr':>7}{'overAb':>8}{'THS':>8}{'thr':>9}")
print("-"*84)
for name,m in R.items():
    thr=f"{m['tuned_thr']:.3f}" if isinstance(m.get('tuned_thr'),(int,float)) else ""
    print(f"{name:<16}{m['cov']:>7.3f}{m['sel_acc']:>8.3f}{m['conf_err']:>7.3f}{m['over_abst']:>8.3f}{m['THS']:>8.2f}{thr:>9}")
print("="*84)
json.dump(R,open(f"{DATA_DIR}/hotpot_clean_metrics_v2.json","w"),indent=2)
print("saved -> hotpot_clean_metrics_v2.json")
print("  P(True) was cov~0.40 THS~29 | logprob cov~0.93 THS~32 | SemEntropy cov~0.80 THS~48")
print("  If P(True)/logprob now ABSTAIN again (cov<1.0), the grid fix worked.")

clean base 0.480  n_base_correct=96

R-Tuning records: eval_rtuning200_progress.json (n=200, keys=['kind', 'ans', 'correct', 'response'])

HotpotQA — UNIFORM RECOMPUTE v2 @ clean base 0.480  (n_base_correct=96)
method              cov  selAcc   cErr  overAb     THS      thr
------------------------------------------------------------------------------------
P(True)           0.405   0.864  0.055   0.438   29.92    0.940
logprob           0.930   0.683  0.295   0.031   36.27   -0.423
verbalized        0.805   0.752  0.200   0.104   42.04    0.250
SemEntropy        0.795   0.811  0.150   0.062   50.65    2.164
CRaFT             0.600   0.833  0.100   0.083   40.77         
TruthRL-ob        0.910   0.681  0.290   0.052   35.23         
TruthRL-cb        0.620   0.524  0.295   0.417    5.27         
R-Tuning          0.215   0.860  0.030   0.740   15.73         
saved -> hotpot_clean_metrics_v2.json

Sanity vs your earlier locked numbers (old 0.515 base, so not identical, but ballpark):
 

In [ ]:

import os, json, numpy as np
D="/content/drive/MyDrive/hedge_run"
def load(p): return json.load(open(p)) if os.path.exists(p) else None

BASE=0.440
FAIR=json.load(open(f"{D}/2wiki_base_point_fair.json"))
base_correct_q={int(k):bool(v) for k,v in FAIR["per_q"].items()}
n_bc=sum(base_correct_q.values()); N=200
assert n_bc==88, f"expected 88 got {n_bc}"

NON_ANSWER=["cannot determine","cannot be determined","could not determine","does not allow",
    "we cannot","do not have","information provided does not","there is no movie",
    "no movie in the given","cannot definitively","does not provide","not allow us to determine",
    "does not contain","cannot be precisely","not confident","cannot verify","unable to determine",
    "insufficient","does not specify","cannot answer","don't know","do not know"]
def is_nonanswer(a): a=str(a).lower(); return any(m in a for m in NON_ANSWER)
def ths(cc,cw,base=BASE): return ((cc/N)*(1-base)-(cw/N)*base)/(1-base)*100
def pack(cc,cw,ab,over):
    return dict(cov=(cc+cw)/N,selAcc=cc/(cc+cw) if cc+cw else 0,cErr=cw/N,
                overAb=over/n_bc,THS=ths(cc,cw),cc=cc,cw=cw,ab=ab)

def from_kind(recs):
    cc=cw=ab=over=0
    for x in recs:
        k=x.get("kind","")
        if "abstain" in k or k=="stop":
            ab+=1;
            if base_correct_q.get(int(x.get("qi",x.get("question_index",-1))),False): over+=1
        elif k=="commit_correct": cc+=1
        elif k=="commit_wrong": cw+=1
        elif k=="commit":
            if x.get("correct"): cc+=1
            else: cw+=1
    return pack(cc,cw,ab,over)

def from_commit_correct(prog):
    cc=cw=ab=over=0
    for k,v in prog.items():
        if not str(k).lstrip("-").isdigit() or not isinstance(v,dict): continue
        qi=int(k)
        if v.get("kind")=="commit" and not is_nonanswer(v.get("ans","")):
            if v.get("correct"): cc+=1
            else: cw+=1
        else:
            ab+=1
            if base_correct_q.get(qi,False): over+=1
    return pack(cc,cw,ab,over)

def tune_signal(val,test,key,abstain_if_greater):
    """bounded grid over distinct val values excluding the extreme (no commit-all)."""
    vals=sorted(set(x[key] for x in val if x.get(key) is not None))
    if len(vals)<2: return None

    grid = [v for v in vals if v>vals[0]] if not abstain_if_greater else [v for v in vals if v<vals[-1]]
    def ev(recs,thr):
        cc=cw=ab=over=0
        for x in recs:
            s=x.get(key); qi=int(x["qi"])
            nonans=is_nonanswer(x.get("answer",x.get("ans","")))
            commit = (s is not None) and (not nonans) and ((s<=thr) if abstain_if_greater else (s>=thr))
            if commit:
                if x.get("correct"): cc+=1
                else: cw+=1
            else:
                ab+=1
                if base_correct_q.get(qi,False): over+=1
        return pack(cc,cw,ab,over)
    best=max((dict(ev(val,t),thr=t) for t in grid), key=lambda d:d["THS"])
    r=ev(test,best["thr"]); r["thr"]=best["thr"]; return r

def from_rule(prog, entail_thr=0.40, doubt_thr=1.0, con_knee=0.5):
    cc=cw=ab=over=0
    for k,v in prog.items():
        if not str(k).lstrip("-").isdigit() or not isinstance(v,dict): continue
        qi=int(k); a=float(v.get("a",0.0)); mc=float(v.get("max_con",0.0))
        nonans=v.get("nonans",is_nonanswer(v.get("ans","")))
        committed=(v.get("kind")=="commit") and not nonans
        if committed:
            doubt=max(0.0,mc-con_knee)+2.0*max(0.0,entail_thr-a)
            if doubt>doubt_thr: committed=False
        if committed:
            if v.get("correct"): cc+=1
            else: cw+=1
        else:
            ab+=1
            if base_correct_q.get(qi,False): over+=1
    return pack(cc,cw,ab,over)

rows=[]

for p in [f"{D}/truthrl_2wiki_ob/eval.json"]:
    r=load(p)
    if r: rows.append(("TruthRL","RLVR",from_kind(r))); break

r=load(f"{D}/baseline_craft_2wiki_fixed/craft_eval.json")
if r: rows.append(("CRaFT (fixed)","SFT-refusal",from_kind(r)))

r=load(f"{D}/rtuning_2wiki_eval_v2_progress.json")
if r: rows.append(("R-Tuning","SFT-refusal",from_commit_correct(r)))

for name,folder,key,ag in [("P(True)","baseline_ptrue_2wiki","p_true",False),
                            ("Semantic Entropy","baseline_semantic_entropy_2wiki","entropy",True),
                            ("Logprob","baseline_logprob_verbalized_2wiki","logprob",False),
                            ("Verbalized","baseline_logprob_verbalized_2wiki","verbalized",False)]:
    val=load(f"{D}/{folder}/val_records.json"); test=load(f"{D}/{folder}/test_records.json")
    if val and test:
        m=tune_signal(val,test,key,ag)
        if m: rows.append((name,"calibration" if name!="Semantic Entropy" else "consistency",m))

prog=load(f"{D}/decision_rule_2wiki_progress.json")
if prog:
    prog=prog.get("decision_rule_gated_weighted",prog) if isinstance(prog,dict) else prog
    rows.append(("Rule (zero-shot)","ours",from_rule(prog)))


print(f"n_base_correct={n_bc}  base={BASE}\n")
print(f"{'method':<20}{'family':<14}{'cov':>7}{'selAcc':>8}{'cErr':>7}{'overAb':>8}{'THS':>8}")
print("-"*72)
anchors={"Rule (zero-shot)":21.50,"TruthRL":30.93}
ok=True
for name,fam,m in sorted(rows,key=lambda r:r[2]["THS"]):
    flag=""
    if name in anchors and abs(m["THS"]-anchors[name])>1.0: flag="  ** ANCHOR OFF **"; ok=False
    print(f"{name:<20}{fam:<14}{m['cov']:>7.3f}{m['selAcc']:>8.3f}{m['cErr']:>7.3f}{m['overAb']:>8.3f}{m['THS']:>8.2f}{flag}")
print("-"*72)
print("ANCHOR", "PASSED ✓ (Rule 21.50, TruthRL 30.93 reproduced)" if ok else "OFF — investigate")
json.dump({n:m for n,f,m in rows}, open(f"{D}/all_metrics_2wiki_clean.json","w"), indent=2)
print("saved -> all_metrics_2wiki_clean.json")
print("\nNOTE: CRaFT now the FIXED version (expect ~16.79, replacing buggy 8.25).")
print("R-Tuning is v2 (expect ~12-14). Calibration re-tuned with bounded grid.")

n_base_correct=88  base=0.44

method              family            cov  selAcc   cErr  overAb     THS
------------------------------------------------------------------------
R-Tuning            SFT-refusal     0.390   0.641  0.140   0.375   14.00
Logprob             calibration     0.490   0.602  0.195   0.307   14.18
P(True)             calibration     0.520   0.615  0.200   0.295   16.29
CRaFT (fixed)       SFT-refusal     0.270   0.796  0.055   0.489   17.18
Semantic Entropy    consistency     0.555   0.631  0.205   0.295   18.89
Verbalized          calibration     0.580   0.629  0.215   0.205   19.61
Rule (zero-shot)    ours            0.465   0.699  0.140   0.250   21.50
TruthRL             RLVR            0.720   0.681  0.230   0.136   30.93
------------------------------------------------------------------------
ANCHOR PASSED ✓ (Rule 21.50, TruthRL 30.93 reproduced)
saved -> all_metrics_2wiki_clean.json

NOTE: CRaFT now the FIXED version (expect ~16.79, replacing buggy 8.25).


In [ ]:


import os
import json
import pandas as pd


D = "/content/drive/MyDrive/hedge_run"



CONFIGS = {

    "HotpotQA": {

        "baseline":
            f"{D}/base_point_fair.json",

        "dpo_results":
            f"{D}/hotpot_dpo_FINAL_CLEAN_v2_test_results.csv",

        "save_summary":
            f"{D}/hotpot_dpo_FINAL_COMMON_BASELINE_summary.json",
    },

    "2WikiMultiHopQA": {

        "baseline":
            f"{D}/2wiki_base_point_fair.json",

        "dpo_results":
            f"{D}/2wiki_dpo_FINAL_CLEAN_test_results.csv",

        "save_summary":
            f"{D}/2wiki_dpo_FINAL_COMMON_BASELINE_summary.json",
    },
}



CORRECT_FIELDS = [
    "base_correct",
    "correct",
    "judge_correct",
    "is_correct",
]

QI_FIELDS = [
    "question_index",
    "qi",
    "index",
]


def extract_correct(record):

    for field in CORRECT_FIELDS:

        if field in record:

            value = record[field]

            if isinstance(value, bool):
                return value

            if isinstance(value, (int, float)):
                return bool(value)

            if isinstance(value, str):

                low = value.strip().lower()

                if low in {
                    "true", "1", "correct", "yes"
                }:
                    return True

                if low in {
                    "false", "0", "incorrect", "wrong", "no"
                }:
                    return False

    return None


def extract_qi(record):

    for field in QI_FIELDS:

        if field in record:

            try:
                return int(record[field])

            except Exception:
                pass

    return None


def load_baseline_map(path):

    assert os.path.exists(path), (
        f"Missing baseline file:\n{path}"
    )

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        data = json.load(f)




    if isinstance(data, list):

        result = {}

        for row in data:

            if not isinstance(row, dict):
                continue

            qi = extract_qi(row)
            correct = extract_correct(row)

            if qi is not None and correct is not None:
                result[qi] = correct

        if result:
            return result



    if isinstance(data, dict):

        result = {}

        for key, value in data.items():

            try:
                qi = int(key)

            except Exception:
                continue



            if isinstance(
                value,
                (bool, int, float),
            ):

                result[qi] = bool(value)
                continue



            if isinstance(value, dict):

                correct = extract_correct(value)

                if correct is not None:
                    result[qi] = correct


        if result:
            return result



        for wrapper_key, value in data.items():

            if isinstance(value, list):

                result = {}

                for row in value:

                    if not isinstance(row, dict):
                        continue

                    qi = extract_qi(row)
                    correct = extract_correct(row)

                    if qi is not None and correct is not None:
                        result[qi] = correct

                if result:
                    return result


            elif isinstance(value, dict):

                result = {}

                for key2, row in value.items():

                    try:
                        qi = int(key2)

                    except Exception:

                        if isinstance(row, dict):
                            qi = extract_qi(row)
                        else:
                            qi = None


                    if isinstance(
                        row,
                        (bool, int, float),
                    ):

                        correct = bool(row)

                    elif isinstance(row, dict):

                        correct = extract_correct(row)

                    else:

                        correct = None


                    if qi is not None and correct is not None:
                        result[qi] = correct


                if result:
                    return result


    raise RuntimeError(
        f"Could not identify question-level correctness "
        f"inside baseline file:\n{path}"
    )


def reconcile(
    dataset_name,
    baseline_path,
    dpo_path,
    expected_base_correct,
):

    print()
    print("=" * 80)
    print(dataset_name)
    print("=" * 80)




    base_map = load_baseline_map(
        baseline_path
    )


    print(
        "Baseline question-level records found:",
        len(base_map)
    )



    assert os.path.exists(dpo_path), (
        f"Missing DPO results:\n{dpo_path}"
    )


    df = pd.read_csv(
        dpo_path
    )


    assert len(df) == 200


    df["question_index"] = (
        df["question_index"]
        .astype(int)
    )

    missing = [

        qi

        for qi in df["question_index"]

        if qi not in base_map
    ]


    if missing:

        raise RuntimeError(
            f"{dataset_name}: "
            f"{len(missing)} DPO questions missing "
            f"from common baseline.\n"
            f"First missing IDs: {missing[:10]}"
        )



    df[
        "common_base_correct"
    ] = df[
        "question_index"
    ].map(
        base_map
    ).astype(bool)


    N = len(df)


    N_BASE_CORRECT = int(
        df[
            "common_base_correct"
        ].sum()
    )


    N_BASE_WRONG = (
        N
        -
        N_BASE_CORRECT
    )


    print()
    print(
        "Common baseline correct:",
        N_BASE_CORRECT
    )

    print(
        "Common baseline wrong:",
        N_BASE_WRONG
    )

    print(
        "Common baseline accuracy:",
        round(
            N_BASE_CORRECT / N,
            4
        )
    )


    assert N_BASE_CORRECT == expected_base_correct, (
        f"{dataset_name}: expected "
        f"{expected_base_correct} common-baseline correct "
        f"but found {N_BASE_CORRECT}."
    )



    answered = (
        df["dpo_action"]
        ==
        "answer"
    )


    abstained = (
        df["dpo_action"]
        ==
        "abstain"
    )


    N_ANSWER = int(
        answered.sum()
    )


    N_ABSTAIN = int(
        abstained.sum()
    )


    N_CORRECT = int(

        df.loc[
            answered,
            "dpo_correct"
        ].astype(int).sum()
    )


    N_WRONG = (
        N_ANSWER
        -
        N_CORRECT
    )



    COVERAGE = (
        N_ANSWER / N
    )


    SELECTIVE_ACCURACY = (
        N_CORRECT / N_ANSWER
    )


    CONFIDENT_ERROR = (
        N_WRONG / N
    )



    N_OVER_ABSTAIN = int(

        (
            df[
                "common_base_correct"
            ]
            &
            abstained
        ).sum()
    )


    OVER_ABSTENTION = (
        N_OVER_ABSTAIN
        /
        N_BASE_CORRECT
    )




    N_CAUGHT_WRONG = int(

        (
            ~df[
                "common_base_correct"
            ]
            &
            abstained
        ).sum()
    )


    ABSTENTION_RECALL = (
        N_CAUGHT_WRONG
        /
        N_BASE_WRONG
    )




    Pc0 = (
        N_BASE_CORRECT / N
    )

    Pw0 = (
        N_BASE_WRONG / N
    )

    Pc = (
        N_CORRECT / N
    )

    Pw = (
        N_WRONG / N
    )


    THS = (
        (
            Pc * Pw0
            -
            Pw * Pc0
        )
        /
        Pw0
    )


    THS100 = (
        THS * 100
    )


    print()
    print("DPO")
    print("-" * 50)

    print(
        "Answered:",
        N_ANSWER
    )

    print(
        "Abstained:",
        N_ABSTAIN
    )

    print(
        "Correct returned:",
        N_CORRECT
    )

    print(
        "Wrong returned:",
        N_WRONG
    )

    print(
        "Coverage:",
        round(
            COVERAGE,
            4
        )
    )

    print(
        "Selective accuracy:",
        round(
            SELECTIVE_ACCURACY,
            4
        )
    )

    print(
        "Confident error:",
        round(
            CONFIDENT_ERROR,
            4
        )
    )

    print()
    print("COMMON-BASELINE METRICS")
    print("-" * 50)

    print(
        "Over-abstention count:",
        N_OVER_ABSTAIN
    )

    print(
        "Over-abstention:",
        round(
            OVER_ABSTENTION,
            4
        )
    )

    print(
        "Baseline-wrong cases abstained:",
        N_CAUGHT_WRONG
    )

    print(
        "Abstention recall:",
        round(
            ABSTENTION_RECALL,
            4
        )
    )

    print(
        "THS x100:",
        round(
            THS100,
            2
        )
    )


    return {

        "dataset":
            dataset_name,

        "n":
            N,

        "common_baseline": {

            "correct":
                N_BASE_CORRECT,

            "wrong":
                N_BASE_WRONG,

            "accuracy":
                N_BASE_CORRECT / N,
        },

        "dpo": {

            "answered":
                N_ANSWER,

            "abstained":
                N_ABSTAIN,

            "correct_returned":
                N_CORRECT,

            "wrong_returned":
                N_WRONG,

            "coverage":
                COVERAGE,

            "selective_accuracy":
                SELECTIVE_ACCURACY,

            "confident_error":
                CONFIDENT_ERROR,

            "over_abstention_count":
                N_OVER_ABSTAIN,

            "over_abstention":
                OVER_ABSTENTION,

            "abstention_recall_count":
                N_CAUGHT_WRONG,

            "abstention_recall":
                ABSTENTION_RECALL,

            "THS":
                THS,

            "THSx100":
                THS100,
        },

        "_df":
            df,
    }



hotpot = reconcile(

    dataset_name=
        "HotpotQA",

    baseline_path=
        CONFIGS[
            "HotpotQA"
        ][
            "baseline"
        ],

    dpo_path=
        CONFIGS[
            "HotpotQA"
        ][
            "dpo_results"
        ],

    expected_base_correct=
        96,
)



wiki = reconcile(

    dataset_name=
        "2WikiMultiHopQA",

    baseline_path=
        CONFIGS[
            "2WikiMultiHopQA"
        ][
            "baseline"
        ],

    dpo_path=
        CONFIGS[
            "2WikiMultiHopQA"
        ][
            "dpo_results"
        ],

    expected_base_correct=
        88,
)



for result, config_name in [

    (
        hotpot,
        "HotpotQA"
    ),

    (
        wiki,
        "2WikiMultiHopQA"
    ),
]:

    clean = {
        k: v
        for k, v in result.items()
        if k != "_df"
    }


    path = CONFIGS[
        config_name
    ][
        "save_summary"
    ]


    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            clean,
            f,
            indent=2,
        )


    print(
        f"\nSaved {config_name}:",
        path
    )



comparison = pd.DataFrame([

    {
        "Dataset":
            "HotpotQA",

        "Base Acc":
            hotpot[
                "common_baseline"
            ][
                "accuracy"
            ],

        "Coverage":
            hotpot["dpo"]["coverage"],

        "Selective Acc":
            hotpot[
                "dpo"
            ][
                "selective_accuracy"
            ],

        "Confident Error":
            hotpot[
                "dpo"
            ][
                "confident_error"
            ],

        "Over-Abstention":
            hotpot[
                "dpo"
            ][
                "over_abstention"
            ],

        "Abstention Recall":
            hotpot[
                "dpo"
            ][
                "abstention_recall"
            ],

        "THS x100":
            hotpot[
                "dpo"
            ][
                "THSx100"
            ],
    },

    {
        "Dataset":
            "2Wiki",

        "Base Acc":
            wiki[
                "common_baseline"
            ][
                "accuracy"
            ],

        "Coverage":
            wiki["dpo"]["coverage"],

        "Selective Acc":
            wiki[
                "dpo"
            ][
                "selective_accuracy"
            ],

        "Confident Error":
            wiki[
                "dpo"
            ][
                "confident_error"
            ],

        "Over-Abstention":
            wiki[
                "dpo"
            ][
                "over_abstention"
            ],

        "Abstention Recall":
            wiki[
                "dpo"
            ][
                "abstention_recall"
            ],

        "THS x100":
            wiki[
                "dpo"
            ][
                "THSx100"
            ],
    },
])


print()
print("=" * 80)
print("FINAL COMMON-BASELINE DPO COMPARISON")
print("=" * 80)

print(
    comparison.round(4).to_string(
        index=False
    )
)


HotpotQA
Baseline question-level records found: 200

Common baseline correct: 96
Common baseline wrong: 104
Common baseline accuracy: 0.48

DPO
--------------------------------------------------
Answered: 114
Abstained: 86
Correct returned: 86
Wrong returned: 28
Coverage: 0.57
Selective accuracy: 0.7544
Confident error: 0.14

COMMON-BASELINE METRICS
--------------------------------------------------
Over-abstention count: 30
Over-abstention: 0.3125
Baseline-wrong cases abstained: 56
Abstention recall: 0.5385
THS x100: 30.08

2WikiMultiHopQA
Baseline question-level records found: 200

Common baseline correct: 88
Common baseline wrong: 112
Common baseline accuracy: 0.44

DPO
--------------------------------------------------
Answered: 89
Abstained: 111
Correct returned: 61
Wrong returned: 28
Coverage: 0.445
Selective accuracy: 0.6854
Confident error: 0.14

COMMON-BASELINE METRICS
--------------------------------------------------
Over-abstention count: 22
Over-abstention: 0.25
Baseline-